# Tratando os dados dp PgAdmin

- @author: Guilherme Nogueira

## Importações

In [1]:
# =========================================================
# BIBLIOTECAS
# =========================================================
import os
import re
import unicodedata
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import text
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text
from sqlalchemy.exc import SQLAlchemyError

from io import BytesIO
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

from sqlalchemy import text


# =========================================================
# FUNÇÕES
# =========================================================
import sys
from pathlib import Path

# PASTA_FUNCOES = Path(r"C:\Users\analy\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\lr-functions\functions")
PASTA_FUNCOES = Path(r"C:\Users\Guilherme\LABOR RURAL\Analytics - Departamento Analytics\TEMP\GUILHERME\SCRIPTS\projetcs\lr-functions\functions")

if not PASTA_FUNCOES.exists():
    raise FileNotFoundError(
        f"Pasta das funções não encontrada:\n{PASTA_FUNCOES}"
    )

if str(PASTA_FUNCOES) not in sys.path:
    sys.path.insert(0, str(PASTA_FUNCOES))

from excel_format import (
    exportar_xlsx_formatado,
    exportar_varias_abas_xlsx,
)

print("Funções de Excel importadas com sucesso.")

Funções de Excel importadas com sucesso.


### Configurações de conexão

In [2]:
# Carrega as variáveis do arquivo .env
load_dotenv()

# Configurações de conexão
PG_HOST = os.getenv("PG_HOST")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DATABASE = os.getenv("PG_DATABASE")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")
PG_SCHEMA = os.getenv("PG_SCHEMA", "analytics_mart")

# Verifica se as configurações obrigatórias foram preenchidas
configuracoes = {
    "PG_HOST": PG_HOST,
    "PG_DATABASE": PG_DATABASE,
    "PG_USER": PG_USER,
    "PG_PASSWORD": PG_PASSWORD,
}

faltantes = [
    nome
    for nome, valor in configuracoes.items()
    if valor is None or str(valor).strip() == ""
]

if faltantes:
    raise ValueError(
        "As seguintes variáveis não foram preenchidas no arquivo .env: "
        + ", ".join(faltantes)
    )


# Criação segura da URL de conexão
url_conexao = URL.create(
    drivername="postgresql+psycopg",
    username=PG_USER,
    password=PG_PASSWORD,
    host=PG_HOST,
    port=PG_PORT,
    database=PG_DATABASE,
)


# Engine de conexão
engine = create_engine(
    url_conexao,
    pool_pre_ping=True,
)

### Testar a conexão

In [ ]:
try:
    with engine.connect() as conexao:
        resultado = conexao.execute(
            text(
                """
                SELECT
                    current_database() AS banco,
                    current_user AS usuario,
                    current_schema() AS schema_atual,
                    version() AS versao
                """
            )
        ).mappings().one()
    
    print("Conexão realizada com sucesso!")
    print(f"Banco: {resultado['banco']}")
    print(f"Usuário: {resultado['usuario']}")
    print(f"Schema atual: {resultado['schema_atual']}")

except SQLAlchemyError as erro:
    print("Não foi possível conectar ao PostgreSQL.")
    raise erro

### Configuração das Views

In [ ]:
# ============================================================
# CONFIGURAÇÕES DAS VIEWS
# ============================================================

# Chaves usadas nos merges
CHAVES = ["id_property", "reference_month"]

# Período analisado
DATA_INICIAL = pd.Timestamp("2023-01-01")

# Primeiro dia do mês atual
DATA_FINAL = (pd.Timestamp.today().to_period("M").to_timestamp())

# View principal da tabela final
VIEW_BASE = "vw_revenue"

# Views que serão adicionadas à view principal
VIEWS_MERGE = [
    "vw_cattle",
    "vw_expense",
    "vw_feeding",
    "vw_labor",
    "vw_own_milk",
    "mvw_area_land_summary",
    "mvw_asset_payment_history",
    "vw_dairy_production_system_monthly",
]

# Lista completa de views autorizadas para importação
views = list(dict.fromkeys( [VIEW_BASE] + VIEWS_MERGE ))

# ============================================================
# CONFIGURAÇÃO ESPECÍFICA DA ÁREA ATIVA
# ============================================================

COLUNAS_AREA_ATIVA = [
    "id_property",
    "reference_month",
    "hectares_owned_benfeitorias_estradas",
    "hectares_owned_app_reserva_legal",
    "hectares_owned_forrageiras",
    "hectares_rented_benfeitorias_estradas",
    "hectares_rented_app_reserva_legal",
    "hectares_rented_forrageiras",
    "raw_land_value_benfeitorias_estradas",
    "raw_land_value_app_reserva_legal",
    "raw_land_value_forrageiras",
]

# ============================================================
# FUNÇÃO DE IMPORTAÇÃO
# ============================================================

def importar_view(nome_view: str, engine, schema: str, ordenar_por: str | None = None,) -> pd.DataFrame:
    """
    Importa uma view PostgreSQL para um DataFrame.

    A view precisa estar cadastrada na lista `views`. 
    A ordenação é feita no pandas somente quando a coluna informada existir.
    """

    if nome_view not in views:
        raise ValueError( f"View não autorizada: {nome_view}" )

    print(f"Importando {schema}.{nome_view}...")
    
    consulta = text(f''' SELECT * FROM "{schema}"."{nome_view}"; ''')
    
    df = pd.read_sql_query(sql=consulta, con=engine)

    if (ordenar_por is not None and ordenar_por in df.columns):
        df = (df.sort_values(ordenar_por).reset_index(drop=True) )

    return df


# ============================================================
# CONFERÊNCIA DAS CONFIGURAÇÕES
# ============================================================

print(f"View principal: {VIEW_BASE}")

print("\nViews usadas nos merges:")
for nome_view in VIEWS_MERGE:
    print(f"- {nome_view}")

print("\nViews autorizadas para importação:")
for nome_view in views:
    print(f"- {nome_view}")

print(f"\nPeríodo: " f"{DATA_INICIAL:%Y-%m} a {DATA_FINAL:%Y-%m}" )

### Criar Driver Supabase

In [ ]:
from supabase import create_client, Client

service_key = os.getenv('SUPABASE_SERVICE_KEY')
print("Chave carregada?", service_key is not None)

# URL do Projeto
project_url = 'https://mrjrkkbecjyzzwkvouxx.supabase.co'

# Acesso ao cliente
global supabase
supabase: Client = create_client(project_url, service_key)
print("Supabase conectado!")

## IGPDI

Ações básicas da função abaixo:
1. Verifica se há recente atualização do IGPDI que não está na base.
2. Se houver, baixa nova planilha, trata e cria as colunas e realiza o upload no supabase.
3. Se não houver, traz a base de dados do supabase

In [ ]:
URL_IGPDI = "https://sindusconpr.com.br/igp-di-fgv-308-p/"

HEADERS_IGPDI = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/150.0.0.0 Safari/537.36"
    )
}


def localizar_link_igpdi(timeout=30):
    """
    Acessa a página do Sinduscon-PR e localiza automaticamente o link de download da série histórica do IGP-DI.
    """
    resposta = requests.get(
        URL_IGPDI,
        headers=HEADERS_IGPDI,
        timeout=timeout
    )
    resposta.raise_for_status()

    soup = BeautifulSoup(resposta.text, "html.parser")

    # Busca prioritária: botão DOWNLOAD dentro da linha do IGP-DI
    for link in soup.select("a[href]"):
        texto_link = link.get_text(" ", strip=True).upper()

        linha = link.find_parent("tr")
        contexto = (
            linha.get_text(" ", strip=True).upper()
            if linha is not None
            else link.parent.get_text(" ", strip=True).upper()
        )

        if "DOWNLOAD" in texto_link and "IGP" in contexto:
            return urljoin(URL_IGPDI, link["href"])

    # Busca alternativa pelos links de download da página
    for link in soup.select("a[href]"):
        href = link.get("href", "")

        if "/download/" in href:
            return urljoin(URL_IGPDI, href)

    raise RuntimeError(
        "Não foi possível localizar o arquivo XLSX do IGP-DI na página."
    )


def carregar_igpdi_supabase():
    """
    Carrega a tabela atual do IGP-DI armazenada no Supabase.
    """
    response = (
        supabase
        .table("tab_igpdi")
        .select("data,igpdi,igpdi_atual,deflator")
        .execute()
    )

    df = pd.DataFrame(response.data or [])

    if df.empty:
        return df

    df["data"] = pd.to_datetime(df["data"], errors="coerce")

    for coluna in ["igpdi", "igpdi_atual", "deflator"]:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

    return (
        df
        .dropna(subset=["data"])
        .sort_values("data")
        .reset_index(drop=True)
    )


def series_igpdi_iguais(df_site, df_supabase):
    """
    Verifica se a série do site é igual à série do Supabase.
    """
    if df_site.empty or df_supabase.empty:
        return False

    site = (
        df_site[["data", "igpdi"]]
        .dropna(subset=["data"])
        .sort_values("data")
        .reset_index(drop=True)
    )

    supa = (
        df_supabase[["data", "igpdi"]]
        .dropna(subset=["data"])
        .sort_values("data")
        .reset_index(drop=True)
    )

    if len(site) != len(supa):
        return False

    mesmas_datas = site["data"].equals(supa["data"])

    mesmos_valores = np.allclose(
        site["igpdi"].to_numpy(dtype=float),
        supa["igpdi"].to_numpy(dtype=float),
        equal_nan=True
    )

    return mesmas_datas and mesmos_valores


def baixar_dados_igpdi(timeout=30):
    """
    Baixa a planilha do IGP-DI diretamente, sem Selenium.

    Se a série estiver igual à armazenada no Supabase, retorna os dados do Supabase.

    Se houver alteração, recalcula o deflator e atualiza toda a tabela no Supabase.
    """
    # 1. Localizar o arquivo
    link_download = localizar_link_igpdi(timeout=timeout)

    # 2. Baixar o XLSX diretamente
    resposta = requests.get(
        link_download,
        headers={
            **HEADERS_IGPDI,
            "Referer": URL_IGPDI
        },
        timeout=timeout
    )
    resposta.raise_for_status()

    # Arquivos XLSX são arquivos ZIP e normalmente começam com PK
    if not resposta.content.startswith(b"PK"):
        raise RuntimeError(
            "O conteúdo baixado não parece ser um arquivo XLSX válido."
        )

    # 3. Ler e tratar sem salvar na pasta Downloads
    arquivo_memoria = BytesIO(resposta.content)

    # A Plan1 possui três linhas de título antes da série histórica.
    df_site = pd.read_excel(
        arquivo_memoria,
        sheet_name="Plan1",
        header=None,
        skiprows=3,
        usecols=[0, 1],
        names=["data", "igpdi"],
    )

    df_site["data"] = pd.to_datetime(df_site["data"], errors="coerce")
    df_site["igpdi"] = pd.to_numeric(df_site["igpdi"], errors="coerce")

    df_site = (
        df_site
        .dropna(subset=["data", "igpdi"])
        .sort_values("data")
        .drop_duplicates(subset=["data"], keep="last")
        .reset_index(drop=True)
    )

    if df_site.empty:
        raise ValueError("Nenhum registro válido de IGP-DI foi encontrado na planilha.")

    # Mantém a convenção existente: deflator = índice do mês / índice mais recente.
    igpdi_atual = df_site["igpdi"].iloc[-1]
    df_site["igpdi_atual"] = igpdi_atual
    df_site["deflator"] = df_site["igpdi"] / igpdi_atual

    # 4. Consultar Supabase
    try:
        df_supabase = carregar_igpdi_supabase()
    except Exception as erro:
        print(f"⚠️ Não foi possível consultar o Supabase: {erro}")
        df_supabase = pd.DataFrame()

    # 5. Retornar Supabase se não houver alteração
    if series_igpdi_iguais(df_site, df_supabase):
        print(f"✅ IGP-DI já está atualizado no Supabase. Último mês: {df_supabase['data'].max():%m/%Y}" )

        return df_supabase

    # 6. Preparar os registros para envio
    df_upload = df_site.copy()
    df_upload["data"] = df_upload["data"].dt.strftime("%Y-%m-%d")
    df_upload = ( df_upload .astype(object) .where(pd.notna(df_upload), None) )

    registros = df_upload.to_dict("records")

    # 7. Atualizar toda a tabela porque o deflator histórico muda
    try:
        (supabase.table("tab_igpdi").delete().gte("data", "1900-01-01").execute())
        (supabase .table("tab_igpdi").insert(registros).execute())
        print(f"✅ Supabase atualizado com {len(registros)} registros. Último mês: {df_site['data'].max():%m/%Y}")

    except Exception as erro:
        raise RuntimeError( f"Erro ao atualizar a tabela tab_igpdi: {erro}" ) from erro

    return df_site

df_igpdi = baixar_dados_igpdi()

## Indicadores Mensais

In [ ]:
consulta_conexao = text("""
    SELECT
        current_database() AS banco_atual,
        current_user AS usuario_atual,
        current_schema() AS schema_atual,
        current_setting('search_path') AS search_path,
        inet_server_addr() AS endereco_servidor,
        inet_server_port() AS porta_servidor;
""")

with engine.connect() as conexao:
    diagnostico_conexao = pd.read_sql_query(consulta_conexao, conexao)

display(diagnostico_conexao)

#### Importação individual das views

In [ ]:
# Cada view recebe um DataFrame próprio para permitir tratamentos específicos.
df_revenue                         = importar_view(nome_view="vw_revenue",                         engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_cattle                          = importar_view(nome_view="vw_cattle",                          engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_expense                         = importar_view(nome_view="vw_expense",                         engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_feeding                         = importar_view(nome_view="vw_feeding",                         engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_labor                           = importar_view(nome_view="vw_labor",                           engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_own_milk                        = importar_view(nome_view="vw_own_milk",                        engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_area                            = importar_view(nome_view="mvw_area_land_summary",              engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_asset_payment_history           = importar_view(nome_view="mvw_asset_payment_history",          engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")
df_dairy_production_system_monthly = importar_view(nome_view="vw_dairy_production_system_monthly", engine=engine, schema=PG_SCHEMA, ordenar_por="id_property")

In [ ]:
# ============================================================
# ATIVOS: CORRIGIR CADA ITEM E DEPOIS AGREGAR POR MÊS
# ============================================================

colunas_necessarias_ativos = [
    "id_property",
    "classification",
    "acquired_at",
    "reference_month",
    "monthly_depreciation",
    "monthly_average_capital_stock",
]

colunas_ausentes_ativos = [
    coluna
    for coluna in colunas_necessarias_ativos
    if coluna not in df_asset_payment_history.columns
]

if colunas_ausentes_ativos:
    raise KeyError(
        "Colunas ausentes em df_asset_payment_history: "
        f"{colunas_ausentes_ativos}"
    )

df_assets = df_asset_payment_history.copy()

df_assets["acquisition_month"] = (
    pd.to_datetime(df_assets["acquired_at"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

df_assets["reference_month"] = (
    pd.to_datetime(df_assets["reference_month"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Uma linha por mês na série do IGP-DI.
df_igpdi_assets = df_igpdi[["data", "igpdi"]].copy()
df_igpdi_assets["data"]  = (pd.to_datetime(df_igpdi_assets["data"], errors="coerce") .dt.to_period("M") .dt.to_timestamp())
df_igpdi_assets["igpdi"] = pd.to_numeric( df_igpdi_assets["igpdi"], errors="coerce", )
df_igpdi_assets = df_igpdi_assets.dropna(subset=["data", "igpdi"])

if df_igpdi_assets.duplicated("data").any():
    raise ValueError("Existem meses duplicados na série do IGP-DI.")

# Primeiro merge: índice do mês em que o ativo foi adquirido.
df_assets = df_assets.merge(
    df_igpdi_assets.rename(columns={"data": "acquisition_month", "igpdi": "igpdi_acquired"}),
    on="acquisition_month",
    how="left",
    validate="many_to_one",
)

DATA_CORTE_IGPDI = pd.Timestamp("1994-08-01")
mask_antes_serie = df_assets["acquisition_month"] < DATA_CORTE_IGPDI
df_assets.loc[mask_antes_serie, "igpdi_acquired"] = 100.0

# Segundo merge: índice do mês de referência da depreciação.
df_assets = df_assets.merge(
    df_igpdi_assets.rename(columns={"data": "reference_month", "igpdi": "igpdi_month"}),
    on="reference_month",
    how="left",
    validate="many_to_one",
)

sem_igpdi_ativo = (df_assets["igpdi_acquired"].isna() | df_assets["igpdi_month"].isna())

if sem_igpdi_ativo.any():
    print(f"⚠️ Itens sem IGP-DI na aquisição ou no mês de referência; será utilizado fator 1 em {sem_igpdi_ativo.sum():,} linhas." )

igpdi_valido = (df_assets["igpdi_acquired"].gt(0) & df_assets["igpdi_month"].gt(0) )

# Atualiza o valor da data de aquisição para o poder monetário do mês.
df_assets["asset_update_factor"] = 1.0
df_assets.loc[igpdi_valido, "asset_update_factor"] = ( df_assets.loc[igpdi_valido, "igpdi_month"] / df_assets.loc[igpdi_valido, "igpdi_acquired"] )

COLUNAS_MONETARIAS_ATIVOS = [
    "monthly_depreciation",
    "monthly_average_capital_stock",
]

for coluna in COLUNAS_MONETARIAS_ATIVOS:
    df_assets[coluna] = (pd.to_numeric(df_assets[coluna], errors="coerce") * df_assets["asset_update_factor"])

# Normaliza acentos e capitalização para consolidar as classificações.
df_assets["classification_normalized"] = (
    df_assets["classification"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
)

mascara_benfeitorias = (df_assets["classification_normalized"] == "benfeitorias")
mascara_maquinas = (df_assets["classification_normalized"] == "maquinas e equipamentos")

df_assets["monthly_depreciation_benfeitorias"]                     = (df_assets["monthly_depreciation"].where(mascara_benfeitorias, 0))
df_assets["monthly_depreciation_maquinas_e_equipamentos"]          = (df_assets["monthly_depreciation"].where(mascara_maquinas, 0))
df_assets["monthly_average_capital_stock_benfeitorias"]            = (df_assets["monthly_average_capital_stock"].where(mascara_benfeitorias, 0))
df_assets["monthly_average_capital_stock_maquinas_e_equipamentos"] = (df_assets["monthly_average_capital_stock"].where(mascara_maquinas, 0))

COLUNAS_MENSAIS_ATIVOS = [
    "monthly_depreciation_benfeitorias",
    "monthly_depreciation_maquinas_e_equipamentos",
    "monthly_average_capital_stock_benfeitorias",
    "monthly_average_capital_stock_maquinas_e_equipamentos",
]

df_asset_payment_history_monthly = (
    df_assets.loc[df_assets["reference_month"].ge(DATA_INICIAL) & df_assets["reference_month"].le(DATA_FINAL) ]
    .groupby(["id_property", "reference_month"], as_index=False)[ COLUNAS_MENSAIS_ATIVOS ]
    .sum()
    .sort_values(["id_property", "reference_month"])
    .reset_index(drop=True)
)

print( "Ativos corrigidos e agrupados: " f"{df_asset_payment_history_monthly.shape[0]:,} linhas mensais." )

In [ ]:
df_asset_payment_history_monthly.head()

In [ ]:
# Mantém compatibilidade com as funções de validação e merge existentes.
dados_views = {
    "vw_revenue": df_revenue,
    "vw_cattle": df_cattle,
    "vw_expense": df_expense,
    "vw_feeding": df_feeding,
    "vw_labor": df_labor,
    "vw_own_milk": df_own_milk,
    "mvw_area_land_summary": df_area,   # nome atualizado
    "mvw_asset_payment_history": df_asset_payment_history_monthly,
    "vw_dairy_production_system_monthly": df_dairy_production_system_monthly,
}

for nome_view, df_view in dados_views.items():
    print(
        f"{nome_view}: {df_view.shape[0]:,} linhas e "
        f"{df_view.shape[1]:,} colunas."
    )

In [ ]:
CHAVES = ["id_property", "reference_month"]

resumo_duplicidades = []
exemplos_duplicidades = {}

for nome_view, df_original in dados_views.items():

    print(f"Verificando {nome_view}...")

    # Trabalhar com uma cópia para não alterar o dado bruto
    df = df_original.copy()

    # ========================================================
    # TRATAMENTO DA VIEW DE ALIMENTAÇÃO
    # ========================================================
    if nome_view == "vw_feeding":

        # A coluna unit não será utilizada
        df = df.drop(
            columns=["unit"],
            errors="ignore",
        )
    
    # ========================================================
    # CONFERIR SE AS CHAVES EXISTEM
    # ========================================================

    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df.columns
    ]

    if colunas_ausentes:

        resumo_duplicidades.append({
            "view": nome_view,
            "linhas_totais": len(df),
            "linhas_em_chaves_duplicadas": None,
            "chaves_duplicadas": None,
            "chave_unica": False,
            "status": f"Chaves ausentes: {colunas_ausentes}",
        })

        print(f"Não foi possível verificar {nome_view}. Colunas ausentes: {colunas_ausentes}" )

        continue

    # ========================================================
    # PADRONIZAR AS CHAVES
    # ========================================================

    df["id_property"] = ( df["id_property"] .astype("string") .str.strip() )
    df["reference_month"] = ( pd.to_datetime( df["reference_month"], errors="coerce", ) .dt.to_period("M") .dt.to_timestamp() )

    # ========================================================
    # VERIFICAR DUPLICIDADES
    # ========================================================

    mascara_duplicada = df.duplicated( subset=CHAVES, keep=False, )

    df_duplicados = ( df.loc[mascara_duplicada] .sort_values(CHAVES) .copy() )
    quantidade_linhas_duplicadas = len( df_duplicados )
    quantidade_chaves_duplicadas = ( df_duplicados[CHAVES] .drop_duplicates() .shape[0] )
    
    resumo_duplicidades.append({
        "view": nome_view,
        "linhas_totais": len(df),
        "linhas_em_chaves_duplicadas": ( quantidade_linhas_duplicadas ),
        "chaves_duplicadas": ( quantidade_chaves_duplicadas ),
        "chave_unica": ( quantidade_linhas_duplicadas == 0 ),
        "status": ( "OK" if quantidade_linhas_duplicadas == 0 else "Possui duplicidades" ),
    })

    if quantidade_linhas_duplicadas > 0:
        exemplos_duplicidades[nome_view] = ( df_duplicados.head(20) )
        print( f"{nome_view}: " f"{quantidade_chaves_duplicadas:,} " "chaves duplicadas.\n")

    else:
        print( f"{nome_view}: nenhuma duplicidade.\n")


# ============================================================
# RESULTADO
# ============================================================

df_resumo_duplicidades = (
    pd.DataFrame(resumo_duplicidades)
    .sort_values(by=["chave_unica", "view"], ascending=[True, True])
    .reset_index(drop=True)
)

display(df_resumo_duplicidades)

#### Tratando os dados antes de exportar

##### Funções de tratamento das views

In [ ]:
def preparar_view(df: pd.DataFrame, nome_view: str) -> pd.DataFrame:
    """
    Prepara uma view mensal para os merges.
    
    - Padroniza id_property;
    - Padroniza o mês de referência;
    - Trata a view de área ativa;
    - Remove unit da view de alimentação;
    - Remove registros sem chave;
    - Filtra o período;
    - Ordena o resultado.
    """
    
    if df is None:
        raise ValueError( f"A view {nome_view} não foi importada." )

    # Trabalhar com uma cópia para preservar dados_views
    df = df.copy()
    
    # ========================================================
    # TRATAMENTO ESPECÍFICO DA ÁREA ATIVA
    # ========================================================
    if nome_view == "mvw_area_land_summary":

        colunas_hectares_owned = [
            "hectares_owned_benfeitorias_estradas",
            "hectares_owned_app_reserva_legal",
            "hectares_owned_forrageiras",
        ]

        colunas_hectares_rented = [
            "hectares_rented_benfeitorias_estradas",
            "hectares_rented_app_reserva_legal",
            "hectares_rented_forrageiras",
        ]

        colunas_raw_land_value = [
            "raw_land_value_benfeitorias_estradas",
            "raw_land_value_app_reserva_legal",
            "raw_land_value_forrageiras",
        ]

        # Área própria: soma das 3 categorias de hectares próprios
        df["hectares_propria"] = df[colunas_hectares_owned].sum(axis=1, skipna=True)

        # Área da atividade: tudo (próprio + arrendado) exceto Reserva Legal e APP
        df["hectares_atividade"] = (
            df["hectares_owned_benfeitorias_estradas"].fillna(0)
            + df["hectares_owned_forrageiras"].fillna(0)
            + df["hectares_rented_benfeitorias_estradas"].fillna(0)
            + df["hectares_rented_forrageiras"].fillna(0)
        )

        # Área total: soma de todas as 6 colunas (próprio + arrendado, todas categorias)
        df["hectares_total"] = (
            df[colunas_hectares_owned].sum(axis=1, skipna=True)
            + df[colunas_hectares_rented].sum(axis=1, skipna=True)
        )

        # Valor médio da terra: média ponderada pelas hectares próprios de cada categoria
        soma_valor_ponderado = sum(
            df[col_valor].fillna(0) * df[col_hectares].fillna(0)
            for col_valor, col_hectares in zip(colunas_raw_land_value, colunas_hectares_owned)
        )

        df["raw_land_value_medio_ponderado"] = (
            soma_valor_ponderado / df["hectares_propria"].replace(0, pd.NA)
        )
    # ========================================================
    # TRATAMENTO ESPECÍFICO DA ALIMENTAÇÃO
    # ========================================================
    # ========================================================
    # CONFERIR AS CHAVES
    # ========================================================
    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df.columns
    ]

    if colunas_ausentes:
        raise KeyError(f"A view {nome_view} não possui as colunas {colunas_ausentes}.")

    # ========================================================
    # PADRONIZAR ID_PROPERTY
    # ========================================================

    df["id_property"] = (df["id_property"] .astype("string") .str.strip())

    # Transformar texto vazio em ausente
    df["id_property"] = df["id_property"].replace( "", pd.NA)

    # ========================================================
    # PADRONIZAR REFERENCE_MONTH
    # ========================================================

    df["reference_month"] = (
        pd.to_datetime(df["reference_month"], errors="coerce")
        .dt.to_period("M")
        .dt.to_timestamp()
    )

    # ========================================================
    # REMOVER LINHAS SEM CHAVE
    # ========================================================

    registros_sem_chave = ( df[CHAVES] .isna() .any(axis=1) )
    quantidade_sem_chave = int( registros_sem_chave.sum() )

    if quantidade_sem_chave > 0:
        print(f"{nome_view}: removendo {quantidade_sem_chave:,} linhas sem chave." )
        df = df.loc[ ~registros_sem_chave ].copy()

    # ========================================================
    # FILTRAR O PERÍODO
    # ========================================================
    df = df.loc[ df["reference_month"].between(DATA_INICIAL, DATA_FINAL, inclusive="both") ].copy()

    # ========================================================
    # ORGANIZAR O RESULTADO
    # ========================================================
    df = (df.sort_values(CHAVES).reset_index(drop=True))

    return df

def verificar_chave_unica(df: pd.DataFrame, nome_view: str) -> None:
    """
    Verifica se existe mais de uma linha para a mesma combinação
    de id_property e reference_month.

    O processamento é interrompido caso existam duplicidades.
    """

    # Identificar todas as linhas que fazem parte de chaves duplicadas
    mascara_duplicadas = df.duplicated(subset=CHAVES, keep=False)

    df_duplicadas = (df.loc[mascara_duplicadas] .copy())

    if not df_duplicadas.empty:

        quantidade_linhas_duplicadas = len( df_duplicadas )
        quantidade_chaves_duplicadas = ( df_duplicadas[CHAVES] .drop_duplicates() .shape[0] )
        exemplos = ( df_duplicadas[CHAVES] .drop_duplicates() .sort_values(CHAVES) .head(10) )

        raise ValueError(
            f"\nA view {nome_view} possui duplicidades.\n"
            f"Linhas envolvidas: "
            f"{quantidade_linhas_duplicadas:,}\n"
            f"Chaves duplicadas: "
            f"{quantidade_chaves_duplicadas:,}\n\n"
            f"Exemplos:\n"
            f"{exemplos.to_string(index=False)}"
        )

    print(
        f"{nome_view}: chave única confirmada "
        f"em {len(df):,} linhas."
    )

##### Criar DataFrame Final

In [ ]:
# ============================================================
# PREPARAR TODAS AS VIEWS
# ============================================================

dados_preparados = {}

for nome_view, df_bruto in dados_views.items():
    
    print(f"\nPreparando {nome_view}...")

    df_preparado = preparar_view( df=df_bruto, nome_view=nome_view)
    
    verificar_chave_unica( df=df_preparado, nome_view=nome_view)

    dados_preparados[nome_view] = ( df_preparado.copy())

print("\nTodas as views foram preparadas.")

print("\nViews disponíveis em dados_preparados:")

for nome_view in dados_preparados:
    print(f"- {nome_view}")

In [ ]:
df_final = (
    dados_preparados[VIEW_BASE]
    .copy()
    .sort_values(CHAVES)
    .reset_index(drop=True)
)

quantidade_linhas_base = len(df_final)

print(f"\nBase revenue criada com " f"{quantidade_linhas_base:,} linhas.")

print(f"Propriedades: " f"{df_final['id_property'].nunique():,}")

print(f"Período: " f"{df_final['reference_month'].min():%Y-%m} " f"a {df_final['reference_month'].max():%Y-%m}")

##### Realizar Merges

In [ ]:
# ============================================================
# MERGE DAS VIEWS COM A VW_REVENUE
# ============================================================

resumo_merge = []

for nome_view in VIEWS_MERGE:

    print(f"\nAdicionando {nome_view}...")

    # --------------------------------------------------------
    # 1. Conferir se a view foi preparada
    # --------------------------------------------------------

    if nome_view not in dados_preparados:
        raise KeyError(f"A view {nome_view} não foi encontrada " "em dados_preparados.")
    
    df_auxiliar = dados_preparados[nome_view].copy()

    # --------------------------------------------------------
    # 2. Conferir se as chaves existem
    # --------------------------------------------------------

    colunas_ausentes = [
        coluna
        for coluna in CHAVES
        if coluna not in df_auxiliar.columns
    ]

    if colunas_ausentes:
        raise KeyError(
            f"A view {nome_view} não possui as chaves "
            f"{colunas_ausentes}."
        )

    # --------------------------------------------------------
    # 3. Conferir novamente se a chave é única
    # --------------------------------------------------------

    duplicadas_auxiliar = df_auxiliar.duplicated(
        subset=CHAVES,
        keep=False,
    )

    if duplicadas_auxiliar.any():

        exemplos = (
            df_auxiliar.loc[
                duplicadas_auxiliar,
                CHAVES,
            ]
            .drop_duplicates()
            .sort_values(CHAVES)
            .head(10)
        )

        raise ValueError(
            f"A view {nome_view} possui duplicidades.\n\n"
            f"{exemplos.to_string(index=False)}"
        )

    # --------------------------------------------------------
    # 4. Renomear colunas que já existem no df_final
    # --------------------------------------------------------

    import re

    prefixo = re.sub(r"^(vw_|mvw_)", "", nome_view)

    colunas_conflitantes = [
        coluna
        for coluna in df_auxiliar.columns
        if coluna not in CHAVES
        and coluna in df_final.columns
    ]

    if colunas_conflitantes:

        df_auxiliar = df_auxiliar.rename(
            columns={
                coluna: f"{prefixo}_{coluna}"
                for coluna in colunas_conflitantes
            }
        )

        print("Colunas renomeadas:", colunas_conflitantes)

    # --------------------------------------------------------
    # 5. Verificar a cobertura antes do merge
    # --------------------------------------------------------

    cobertura = (
        df_final[CHAVES]
        .merge(
            df_auxiliar[CHAVES],
            on=CHAVES,
            how="left",
            indicator=True,
            validate="one_to_one",
        )
    )

    chaves_encontradas = int(
        cobertura["_merge"]
        .eq("both")
        .sum()
    )

    chaves_sem_correspondencia = int(
        cobertura["_merge"]
        .eq("left_only")
        .sum()
    )

    linhas_antes = len(df_final)

    # --------------------------------------------------------
    # 6. Realizar o left merge
    # --------------------------------------------------------

    df_final = df_final.merge(
        df_auxiliar,
        on=CHAVES,
        how="left",
        validate="one_to_one",
    )

    linhas_depois = len(df_final)

    # --------------------------------------------------------
    # 7. Validar se a quantidade de linhas foi preservada
    # --------------------------------------------------------

    if linhas_antes != linhas_depois:
        raise ValueError(
            f"O merge com {nome_view} alterou a quantidade "
            f"de linhas de {linhas_antes:,} para "
            f"{linhas_depois:,}."
        )

    # --------------------------------------------------------
    # 8. Registrar o resumo
    # --------------------------------------------------------

    cobertura_percentual = round(
        chaves_encontradas / linhas_antes * 100,
        2,
    )

    resumo_merge.append({
        "view": nome_view,
        "linhas_view": len(df_auxiliar),
        "chaves_encontradas": chaves_encontradas,
        "chaves_sem_correspondencia": (
            chaves_sem_correspondencia
        ),
        "cobertura_percentual": cobertura_percentual,
        "colunas_adicionadas": (
            len(df_auxiliar.columns)
            - len(CHAVES)
        ),
        "linhas_antes": linhas_antes,
        "linhas_depois": linhas_depois,
    })

    print( f"Correspondências: " f"{chaves_encontradas:,}" )
    print( f"Sem correspondência: " f"{chaves_sem_correspondencia:,}" )
    print( f"Cobertura: " f"{cobertura_percentual:.2f}%" )
    print( f"Linhas antes/depois: " f"{linhas_antes:,} / {linhas_depois:,}" )


# ============================================================
# ORGANIZAR O RESULTADO FINAL
# ============================================================

df_final = (
    df_final
    .sort_values(CHAVES)
    .reset_index(drop=True)
)


# ============================================================
# VALIDAÇÕES FINAIS
# ============================================================

assert len(df_final) == quantidade_linhas_base, (
    "Os merges alteraram a quantidade de linhas da revenue."
)

assert not df_final.duplicated(CHAVES).any(), (
    "A tabela final possui duplicidades por "
    "id_property e reference_month."
)


# ============================================================
# RESUMO DOS MERGES
# ============================================================

df_resumo_merge = pd.DataFrame(
    resumo_merge
)

print("\nTodos os merges foram concluídos com sucesso.")
print(f"Linhas da base revenue: {quantidade_linhas_base:,}" )
print(f"Linhas da tabela final: {df_final.shape[0]:,}" )
print(f"Colunas da tabela final: {df_final.shape[1]:,}" )
print(f"Duplicidades por propriedade e mês: {df_final.duplicated(CHAVES).sum()}")

display(df_resumo_merge)
display(df_final.head())

In [ ]:
df_final.head()

In [ ]:
df_final.columns.to_list()

##### Deflacionando as colunas necessárias

In [ ]:
# ============================================================
# DEFLACIONAR AS VARIÁVEIS MONETÁRIAS
# ============================================================

# A deflação ocorre antes do Feature Engineering.
df_integrada = df_final.copy()

# ============================================================
# PREPARAR A BASE DO IGP-DI
# ============================================================

# Seleciona somente as colunas necessárias.
df_igpdi_aux = df_igpdi[[ "data", "deflator"] ].copy()

# Padroniza a data do IGP-DI para o primeiro dia de cada mês.
df_igpdi_aux["data"] = (
    pd.to_datetime(df_igpdi_aux["data"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Converte o deflator para formato numérico.
df_igpdi_aux["deflator"] = pd.to_numeric(df_igpdi_aux["deflator"], errors="coerce")

# Remove linhas sem data válida.
df_igpdi_aux = df_igpdi_aux.dropna(subset=["data"])

# Verifica se existem meses duplicados na tabela do IGP-DI.
if df_igpdi_aux.duplicated("data").any():
    meses_duplicados = (
        df_igpdi_aux.loc[df_igpdi_aux.duplicated("data", keep=False), "data"]
        .dt.strftime("%Y-%m")
        .unique()
        .tolist()
    )
    raise ValueError( "Existem meses duplicados na base do IGP-DI: " f"{meses_duplicados}" )


# ============================================================
# PADRONIZAR O MÊS DA BASE INTEGRADA
# ============================================================

df_integrada["reference_month"] = (
    pd.to_datetime(df_integrada["reference_month"], errors="coerce")
    .dt.to_period("M")
    .dt.to_timestamp()
)

# ============================================================
# ADICIONAR O DEFLATOR
# ============================================================

df_integrada = df_integrada.drop(
    columns=["data", "deflator"],
    errors="ignore",
)

df_integrada = df_integrada.merge(
    df_igpdi_aux,
    left_on="reference_month",
    right_on="data",
    how="left",
    validate="many_to_one",
)


# ============================================================
# VALIDAR O DEFLATOR
# ============================================================

meses_sem_deflator = (
    df_integrada.loc[ df_integrada["deflator"].isna(), "reference_month", ]
    .dropna()
    .dt.strftime("%Y-%m")
    .unique()
    .tolist()
)

if meses_sem_deflator:
    print(f"⚠️ Meses sem IGP-DI; o deflator 1 será utilizado: {meses_sem_deflator}" )

# Na ausência de IGP-DI, mantém o valor nominal usando deflator igual a 1.
df_integrada["deflator"] = df_integrada["deflator"].fillna(1.0)

if df_integrada["deflator"].le(0).any():
    raise ValueError( "Foram encontrados valores de deflator iguais ou inferiores a zero." )

# ============================================================
# DEFINIR AS COLUNAS MONETÁRIAS DE RECEITA
# ============================================================

COLUNAS_DEFLACIONAR_RECEITAS = [
    "milk_sold_revenue",         # Receita total da venda de leite.
    "milk_unit_price",           # Preço unitário do leite.
    "received_loans",            # Empréstimos recebidos.
    "animal_sale",               # Receita com venda de animais.
    "other_revenues",            # Outras receitas.
    "price_bonus",               # Bonificação do preço do leite.
    "price_penalty",             # Penalização ou desconto aplicado ao leite.
    "milk_derivatives_revenue",  # Receita total com derivados.
    "unit_price_derivative",     # Preço unitário dos derivados.
    "voluminous_sold",           # Receita com venda de volumoso.
    "concentrated_sold",          
    "surplus_division",          # Receita com divisão de sobras.
]

COLUNAS_DEFLACIONAR_DESPESAS = [
    "general_expenses",
    "advance_payment",
    "administration",
    "land_lease",
    "technical_assistance",
    "animal_purchase",
    "land_purchase",
    "repairs",
    "loan_interest",
    "hormones",
    "taxes_fees",
    "medicines_vaccines",
    "bedding_replacement",
    "reproduction",
    "milk_replacer",
    "milking_material",
    "milk_calves",
    "energy",
    "fuel",
]


COLUNAS_DEFLACIONAR_ANIMAIS = [
    "lactating_cows_value",
    "dry_cows_value",
    "nursing_value",
    "rearing_value",
    "males_value",
    "other_categories_value",
]

COLUNAS_DEFLACIONAR_MAQ_BEN = [
    'monthly_depreciation_benfeitorias',
    'monthly_depreciation_maquinas_e_equipamentos',
    'monthly_average_capital_stock_benfeitorias',
    'monthly_average_capital_stock_maquinas_e_equipamentos'
]

COLUNAS_DEFLACIONAR = (
    COLUNAS_DEFLACIONAR_RECEITAS
    + COLUNAS_DEFLACIONAR_DESPESAS
    + COLUNAS_DEFLACIONAR_MAQ_BEN
    # + COLUNAS_DEFLACIONAR_ANIMAIS
)

# ============================================================
# CONFERIR SE AS COLUNAS EXISTEM
# ============================================================

colunas_ausentes = [
    coluna
    for coluna in COLUNAS_DEFLACIONAR
    if coluna not in df_integrada.columns
]

if colunas_ausentes:
    raise KeyError(f"As seguintes colunas monetárias não foram encontradas: {colunas_ausentes}" )


# ============================================================
# DEFLACIONAR AS COLUNAS
# ============================================================

for coluna in COLUNAS_DEFLACIONAR:

    # Converte o valor monetário para número e substitui a própria coluna.
    df_integrada[coluna] = (
        pd.to_numeric(df_integrada[coluna], errors="coerce")
        / df_integrada["deflator"]
    )

# ============================================================
# ORGANIZAR A BASE
# ============================================================

# Remove a coluna auxiliar de data proveniente do IGP-DI. O deflator é mantido para rastreabilidade.
df_integrada = df_integrada.drop(columns=["data"], errors="ignore", )

# Substitui eventuais infinitos por NaN.
df_integrada[COLUNAS_DEFLACIONAR] = (
    df_integrada[COLUNAS_DEFLACIONAR]
    .replace( [np.inf, -np.inf], np.nan, )
)

# ============================================================
# CONFERÊNCIA
# ============================================================

display(
    df_integrada[
        [
            "id_property",
            "reference_month",
            "deflator",
            "milk_unit_price",
            "milk_sold_revenue",
            "general_expenses",
            "lactating_cows_value",
        ]
    ].head(20)
)

print("Deflação das receitas, despesas e valores dos animais concluída com sucesso.")

##### Feature Engineering

In [ ]:
# ============================================================
# 1. UTILIZAR A BASE INTEGRADA JÁ DEFLACIONADA
# ============================================================

CHAVES = ["id_property", "reference_month"]

linhas_iniciais = len(df_integrada)

print(f"Linhas: {df_integrada.shape[0]:,}")
print(f"Colunas antes das flags: {df_integrada.shape[1]:,}")


# ============================================================
# 2. MAPA DAS VIEWS E COLUNAS DE PRESENÇA
# ============================================================
MAPA_PRESENCA = {
    "vw_cattle": "has_cattle_data",
    "vw_expense": "has_expense_data",
    "vw_feeding": "has_feeding_data",
    "vw_labor": "has_labor_data",
    "vw_own_milk": "has_own_milk_data",
    "mvw_asset_payment_history": "has_asset_data",
    "mvw_area_land_summary": "has_active_area_month",
    "vw_dairy_production_system_monthly": "has_dairy_production_system_data",
}

# A revenue é a base da tabela final. Portanto, todas as linhas possuem revenue.
df_integrada["has_revenue_data"] = 1

# ============================================================
# 3. CRIAR AS FLAGS DE PRESENÇA
# ============================================================
for nome_view, nome_flag in MAPA_PRESENCA.items():

    print(f"Criando indicador de presença: {nome_flag}")

    if nome_view not in dados_preparados:
        raise KeyError( f"A view {nome_view} não foi encontrada em dados_preparados." )

    # Evita duplicação caso a célula seja executada novamente
    df_integrada = df_integrada.drop(
        columns=[nome_flag],
        errors="ignore",
    )

    # Uma linha por propriedade e mês presente na view
    chaves_disponiveis = (
        dados_preparados[nome_view][CHAVES]
        .dropna(subset=CHAVES)
        .drop_duplicates(subset=CHAVES)
        .assign(**{nome_flag: 1})
    )

    linhas_antes = len(df_integrada)

    df_integrada = df_integrada.merge(
        chaves_disponiveis,
        on=CHAVES,
        how="left",
        validate="one_to_one",
    )

    linhas_depois = len(df_integrada)

    if linhas_antes != linhas_depois:
        raise ValueError(
            f"O merge da flag {nome_flag} alterou a quantidade de linhas: "
            f"{linhas_antes:,} para {linhas_depois:,}."
        )

    df_integrada[nome_flag] = (
        df_integrada[nome_flag]
        .fillna(0)
        .astype("int8")
    )

def somar_colunas_preservando_ausencia(df: pd.DataFrame, colunas: list[str]) -> pd.Series:
    """
    Soma as colunas informadas.
    Quando todas as colunas estiverem ausentes na linha, mantém o resultado como NaN em vez de transformar em zero.
    """

    colunas_ausentes = [
        coluna
        for coluna in colunas
        if coluna not in df.columns
    ]
    
    if colunas_ausentes:
        raise KeyError("Colunas necessárias não encontradas: " + ", ".join(colunas_ausentes))

    return df[colunas].sum(axis=1, min_count=1)

# ============================================================
# 4. ORGANIZAR AS COLUNAS DE PRESENÇA
# ============================================================

colunas_presenca = [
    "has_revenue_data",
    "has_cattle_data",
    "has_expense_data",
    "has_feeding_data",
    "has_labor_data",
    "has_own_milk_data",
    "has_asset_data",
    "has_active_area_month",
    "has_dairy_production_system_data"
]


# ============================================================
# 5. VALIDAR O RESULTADO
# ============================================================

assert len(df_integrada) == linhas_iniciais, ("A criação das flags alterou a quantidade de linhas.")

assert not df_integrada.duplicated(CHAVES).any(), ("Foram geradas duplicidades por id_property e reference_month.")

print("\nFlags de presença criadas com sucesso.")
print(f"Linhas finais: {df_integrada.shape[0]:,}")
print(f"Colunas finais: {df_integrada.shape[1]:,}")


# ============================================================
# 6. CALCULAR A COBERTURA DE CADA VIEW
# ============================================================
df_cobertura_views = (
    df_integrada[colunas_presenca]
    .mean()
    .mul(100)
    .round(2)
    .rename("coverage_percentage")
    .rename_axis("source")
    .reset_index()
)

display(df_cobertura_views)

# ============================================================
# 7. CALCULAR INDICADORES PADRÃO
# ============================================================

# Renda do leite consumido
df_integrada['discarded_quantity_milk_revenue']   = df_integrada['discarded_quantity'] * df_integrada['milk_unit_price']
df_integrada['hired_labor_quantity_milk_revenue'] = df_integrada['own_milk_hired_labor_quantity'] * df_integrada['milk_unit_price']
df_integrada['discarded_quantity_milk_revenue']   = df_integrada['discarded_quantity'] * df_integrada['milk_unit_price']
df_integrada['family_labor_milk_revenue']         = df_integrada['own_milk_family_labor_quantity'] * df_integrada['milk_unit_price']
df_integrada['calves_quantity_milk_revenue']      = df_integrada['calves_quantity'] * df_integrada['milk_unit_price']

# Renda do leite
df_integrada['total_milk_revenue'] = df_integrada[[
    'milk_sold_revenue',
    'milk_derivatives_revenue',
    'price_bonus',
    'discarded_quantity_milk_revenue',
    'hired_labor_quantity_milk_revenue',
    'family_labor_milk_revenue',
    'calves_quantity_milk_revenue'
]].sum(axis=1).round(2) - df_integrada['price_penalty']

# Leite produzido
df_integrada['milk_produced'] = df_integrada[[
    'milk_volume_sold',
    'milk_volume_derivatives',
    'discarded_quantity',
    'own_milk_hired_labor_quantity',
    'own_milk_family_labor_quantity',
    'calves_quantity']].sum(axis=1).round(2)

# Renda da Atividade
df_integrada['total_activity_revenue'] = (
    df_integrada[[
        'total_milk_revenue',
        'received_loans',
        'animal_sale',
        'other_revenues',
        'voluminous_sold',
        'concentrated_sold',
        'surplus_division',
    ]].sum(axis=1)
)
# Preço do Leite
df_integrada['milk_revenue_liter'] = df_integrada['total_milk_revenue'] / df_integrada['milk_produced']

# Leite diário
# Quantidade de dias do mês
df_integrada["days_in_month"] = ( df_integrada["reference_month"].dt.days_in_month)

# Produção média diária de leite
df_integrada["milk_daily"] = (df_integrada["milk_produced"] / df_integrada["days_in_month"].replace(0, np.nan))

# Produção diária por vaca em lactação
df_integrada["milk_lactating_cow_day"] = (df_integrada["milk_daily"] / df_integrada["lactating_cows"].replace(0, np.nan))

# Vacas em lactação sobre o total de vacas
df_integrada["lactating_cows_total_cows"] = (df_integrada["lactating_cows"] / df_integrada["total_cows"].replace(0, np.nan)) * 100

# Vacas em lactação sobre o total do rebanho
df_integrada["lactating_cows_total_cattle"] = (df_integrada["lactating_cows"] / df_integrada["total_cattle"].replace(0, np.nan) ) * 100

# Ajustar mão de obra para dias/homem
df_integrada['hired_labor_quantity']  = df_integrada['hired_labor_quantity'] / df_integrada["days_in_month"]
df_integrada['family_labor_quantity'] = df_integrada['family_labor_quantity'] / df_integrada["days_in_month"]

# Quantidade total de mão de obra
df_integrada["total_labor_quantity"] = ( df_integrada[[ "hired_labor_quantity", "family_labor_quantity"]] .sum(axis=1, min_count=1) )

# Produção diária por unidade de mão de obra total
df_integrada["milk_total_labor_day"] = (df_integrada["milk_daily"] / df_integrada["total_labor_quantity"].replace(0, np.nan))

# Vacas em lactação por unidade de mão de obra
df_integrada["lactating_cows_total_labor"] = (df_integrada["lactating_cows"] / df_integrada["total_labor_quantity"].replace(0, np.nan))

# Custo de concentrado e minerais
df_integrada["concentrate_mineral_cost"] = (df_integrada["concentrate_amount_total"].fillna(0) + df_integrada["mineral_amount_total"].fillna(0))

# Custo total da alimentação
df_integrada["feeding_cost"] = (df_integrada["voluminous_amount_total"].fillna(0) + df_integrada["concentrate_amount_total"].fillna(0) + df_integrada["mineral_amount_total"].fillna(0) )

# Quantidade de concentrado 
df_integrada["concentrate_mineral_quantity"] = (df_integrada["concentrate_purchased_quantity"].fillna(0) + df_integrada["mineral_purchased_quantity"].fillna(0))

# Custo da alimentação por litro
df_integrada["feeding_cost_liter"] = (df_integrada["feeding_cost"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo de volumoso por litro
df_integrada["voluminous_cost_liter"] = (df_integrada["voluminous_amount_total"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo de concentrado e minerais por litro
df_integrada["concentrate_mineral_cost_liter"] = (df_integrada["concentrate_mineral_cost"] / df_integrada["milk_produced"].replace(0, np.nan))

# Participação do custo da alimentação no preço do leite
df_integrada["feeding_cost_milk_price"] = (df_integrada["feeding_cost_liter"] / df_integrada["milk_revenue_liter"].replace(0, np.nan) ) * 100

# Custo total da mão de obra
df_integrada["total_labor_expenses"] = (df_integrada[["hired_labor_expenses", "family_labor_expenses"]] .sum(axis=1, min_count=1))

# Custo da mão de obra contratada por litro
df_integrada["hired_labor_cost_liter"] = (df_integrada["hired_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo da mão de obra familiar por litro
df_integrada["family_labor_cost_liter"] = (df_integrada["family_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Custo total da mão de obra por litro
df_integrada["total_labor_cost_liter"] = (df_integrada["total_labor_expenses"] / df_integrada["milk_produced"].replace(0, np.nan))

# Participação da mão de obra na receita do leite
df_integrada["labor_cost_milk_revenue"] = ( df_integrada["total_labor_expenses"] / df_integrada["total_milk_revenue"].replace(0, np.nan) ) * 100

# Agregação das demais despesas operacionais
COLUNAS_OUTRAS_DESPESAS = [
    "milk_replacer",
    "milking_material",
    "reproduction",
    "hormones",
    "medicines_vaccines",
    "technical_assistance",
    "taxes_fees",
    "land_lease",
    "repairs",
    "administration",
    "general_expenses",
    "bedding_replacement",
]

df_integrada["other_operating_expenses"] = (somar_colunas_preservando_ausencia(df=df_integrada, colunas=COLUNAS_OUTRAS_DESPESAS) )

# Estoque de capital de animais
df_integrada["animal_capital_stock"] = (df_integrada["lactating_cows"] * df_integrada["lactating_cows_value"]) + (df_integrada["dry_cows"] * df_integrada["dry_cows_value"]) + (df_integrada["nursing"] * df_integrada["nursing_value"]) + (df_integrada["rearing"] * df_integrada["rearing_value"]) + (df_integrada["males"] * df_integrada["males_value"]) + ((df_integrada["other_categories"] * df_integrada["other_categories_value"])/2)

# Estoque de capital da terra
df_integrada["land_capital_stock"] = df_integrada["hectares_propria"] * df_integrada["raw_land_value_medio_ponderado"]

# Define os componentes do estoque de capital fixo.
COLUNAS_CAPITAL_FIXO = [
    "monthly_average_capital_stock_benfeitorias",
    "monthly_average_capital_stock_maquinas_e_equipamentos",
]

# Soma o estoque médio de benfeitorias com o estoque médio de máquinas e equipamentos.
#
# min_count=2 exige que os dois valores estejam disponíveis.
df_integrada["fixed_capital_stock"] = (
    df_integrada[COLUNAS_CAPITAL_FIXO]
    .sum(
        axis=1,
        min_count=len(COLUNAS_CAPITAL_FIXO),
    )
)


# Estoque de capital total
COLUNAS_CAPITAL_TOTAL = [
    "animal_capital_stock",
    "land_capital_stock",
    "fixed_capital_stock",
]

df_integrada["total_capital_stock"] = (
    df_integrada[COLUNAS_CAPITAL_TOTAL]
    .sum(axis=1)
)

# Calcula o estoque de capital total por litro de produção diária.
#
# Esse indicador é o equivalente novo de:
# estoqueCapitalcomTerra_leiteDiario.
#
# A produção diária igual a zero é substituída por NaN
# para evitar divisão por zero.
df_integrada["total_capital_stock_milk_daily"] = (
    df_integrada["total_capital_stock"]
    / df_integrada["milk_daily"].replace(0, np.nan)
)


# Substitui eventuais resultados infinitos por NaN.
df_integrada[
    [
        "total_capital_stock",
        "total_capital_stock_milk_daily",
    ]
] = (
    df_integrada[
        [
            "total_capital_stock",
            "total_capital_stock_milk_daily",
        ]
    ]
    .replace([np.inf, -np.inf], np.nan)
)


# ============================================================
# INDICADORES DE ÁREA
# ============================================================

# Vacas em lactação por hectare de atividade
df_integrada["lactating_cows_hectare_activity"] = (df_integrada["lactating_cows"] / df_integrada["hectares_atividade"].replace(0, np.nan))

# Percentual da área arrendada
df_integrada["hectares_arrendada"] = df_integrada[[
    "hectares_rented_benfeitorias_estradas",
    "hectares_rented_app_reserva_legal",
    "hectares_rented_forrageiras",
]].sum(axis=1, skipna=True)

df_integrada["rented_area_percentage"] = (
    df_integrada["hectares_arrendada"] / df_integrada["hectares_total"].replace(0, np.nan) * 100
)

df_integrada.head(20)

In [ ]:
df_integrada.columns.tolist()  

In [ ]:
df_integrada[['animal_capital_stock']].head()

##### Importar Dimensão Produtor

In [ ]:
from sqlalchemy import text

# ============================================================
# IMPORTAR APENAS AS COLUNAS DIMENSIONAIS NECESSÁRIAS
# ============================================================
consulta_dim_property = text(
    """
    SELECT
        id_property,
        property_name,
        labor_rural_code,
        entrepreneur_name,
        agroindustry_name,
        dairy_region,
        property_status
    FROM analytics_mart.vw_dim_property;
    """
)

df_dim_property = pd.read_sql_query(consulta_dim_property, con=engine)

# Padronizar a chave
df_dim_property["id_property"] = (df_dim_property["id_property"].astype("string").str.strip())

# Remover linhas sem chave
df_dim_property = (df_dim_property.dropna(subset=["id_property"]).reset_index(drop=True) )

# Validar uma linha por propriedade
if df_dim_property.duplicated("id_property").any():
    raise ValueError( "A dimensão possui mais de uma linha por id_property." )

# Torna o merge idempotente: remove versões anteriores das colunas dimensionais.
colunas_dimensionais = [
    coluna
    for coluna in df_dim_property.columns
    if coluna != "id_property"
]

colunas_dimensionais_antigas = [
    nome
    for coluna in colunas_dimensionais
    for nome in (coluna, f"{coluna}_x", f"{coluna}_y")
    if nome in df_integrada.columns
]

df_integrada = df_integrada.drop(
    columns=colunas_dimensionais_antigas,
    errors="ignore",
)

# Merge dimensional
linhas_antes = len(df_integrada)

df_integrada = df_integrada.merge(
    df_dim_property,
    on="id_property",
    how="left",
    validate="many_to_one",
)

if len(df_integrada) != linhas_antes:
    raise ValueError( "O merge dimensional alterou a quantidade de linhas." )

print("Merge dimensional concluído.")
print(f"Linhas: {df_integrada.shape[0]:,}")
print(f"Colunas: {df_integrada.shape[1]:,}")

df_integrada.head()

# ============================================================
# CONSULTOR: IMPORTAR AQUI, MAS NÃO MESCLAR AGORA
# ============================================================
# Uma propriedade pode ter mais de um consultor em dim_consultor_propriedade.
# Se mesclássemos isso em df_integrada agora (grão mensal), cada mês de uma
# fazenda com 2+ consultores viraria 2+ linhas ANTES da Feature Engineering e
# da janela anual — o que faria toda soma anual (COE, receita, produção etc.)
# ser contada 2x, 3x etc. Por isso só importamos aqui; o merge (que duplica de
# propósito) acontece só no final, em df_anuais, depois que as somas já estão
# fechadas — ver célula anuais-indicadores.
# A view dim_consultor_propriedade está em analytics_mart.
consulta_dim_consultor = text(
    """
    SELECT
        id_consultor,
        nome_consultor,
        id_property
    FROM analytics_mart.dim_consultor_propriedade;
    """
)

df_dim_consultor = pd.read_sql_query(consulta_dim_consultor, con=engine)
df_dim_consultor["id_property"] = (df_dim_consultor["id_property"].astype("string").str.strip())
df_dim_consultor = (df_dim_consultor.dropna(subset=["id_property"]).reset_index(drop=True) )

print(f"Consultores importados: {df_dim_consultor.shape[0]:,} linhas.")
print(f"Propriedades com mais de um consultor: {df_dim_consultor['id_property'].duplicated().sum():,}")

##### Regras de consitência dos Indicadores Mensais

In [ ]:
# ============================================================
# REGRAS MENSAIS DE CONSISTÊNCIA
# ============================================================
df_consistencia = df_integrada.copy(deep=True)

# ============================================================
# 1. DEFINIR AS COLUNAS NECESSÁRIAS
# ============================================================

# Lista das colunas que precisam existir antes de calcular os indicadores e as regras de consistência.
COLUNAS_NECESSARIAS_CONSISTENCIA = [
    "ccs",
    "cpp",
    "fat",
    "protein",
    "lactating_cows",
    "lactating_cows_total_cows",
    "lactating_cows_total_cattle",
    "milk_daily",
    "milk_total_labor_day",
    "lactating_cows_total_labor",
    "feeding_cost_milk_price",
    "voluminous_cost_liter",
    "concentrate_mineral_cost_liter",
    "hired_labor_cost_liter",
    # "total_capital_stock_milk_daily"
]


# Verifica quais colunas da lista acima não existem na df_consistencia.
colunas_ausentes = [
    coluna
    for coluna in COLUNAS_NECESSARIAS_CONSISTENCIA
    if coluna not in df_consistencia.columns
]

# Interrompe a execução caso alguma coluna obrigatória esteja ausente.
if colunas_ausentes:
    raise KeyError( "As seguintes colunas necessárias para a consistência " f"não foram encontradas: {colunas_ausentes}" )


# ============================================================
# 2. GARANTIR QUE AS COLUNAS SEJAM NUMÉRICAS
# ============================================================

# Percorre cada coluna usada nas regras de consistência.
for coluna in COLUNAS_NECESSARIAS_CONSISTENCIA:
    
    # Converte a coluna para número.
    # Valores que não puderem ser convertidos serão transformados em NaN.
    df_consistencia[coluna] = pd.to_numeric( df_consistencia[coluna], errors="coerce", )


# Substitui valores infinitos positivos e negativos por NaN.
# Isso impede que divisões inválidas sejam classificadas como consistentes.
df_consistencia[COLUNAS_NECESSARIAS_CONSISTENCIA] = (
    df_consistencia[COLUNAS_NECESSARIAS_CONSISTENCIA]
    .replace([np.inf, -np.inf], np.nan)
)

# ============================================================
# 4. FUNÇÃO PARA PADRONIZAR AS REGRAS
# ============================================================

def preparar_regra_consistencia(condicao: pd.Series) -> pd.Series:
    """
    Converte o resultado de uma regra em verdadeiro ou falso.

    Valores ausentes são classificados como False, ou seja,
    o critério não é considerado atendido quando não há informação.
    """

    # Substitui resultados ausentes por False.
    condicao = condicao.fillna(False)

    # Garante que o resultado final tenha tipo booleano.
    return condicao.astype(bool)


# ============================================================
# 5. QUALIDADE DO LEITE
# ============================================================

# CCS é considerada consistente quando for maior que 50.
df_consistencia["cons_ccs"] = preparar_regra_consistencia( df_consistencia["ccs"].gt(50) )

# CPP é considerada consistente quando for maior que 1.
df_consistencia["cons_cpp"] = preparar_regra_consistencia( df_consistencia["cpp"].gt(1) )

# Gordura é consistente quando estiver acima de 2,5 e abaixo de 5,5.
df_consistencia["cons_fat"] = preparar_regra_consistencia( df_consistencia["fat"].gt(2.5) & df_consistencia["fat"].lt(5.5) )

# Proteína é consistente quando estiver acima de 2,4 e abaixo de 4,5.
df_consistencia["cons_protein"] = preparar_regra_consistencia( df_consistencia["protein"].gt(2.4) & df_consistencia["protein"].lt(4.5) )


# ============================================================
# 6. ESTRUTURA DO REBANHO
# ============================================================

# Na base nova, lactating_cows_total_cows está em percentual. 
# O limite antigo de 0,20 a 0,99 corresponde agora a 20% a 99%.
df_consistencia["cons_lactating_cows_total_cows"] = (
    preparar_regra_consistencia(
        df_consistencia["lactating_cows_total_cows"].gt(20)
        & df_consistencia["lactating_cows_total_cows"].lt(99)
    )
)

# Na base nova, lactating_cows_total_cattle também está em percentual. 
# O limite antigo de 0,15 a 0,99 corresponde agora a 15% a 99%.
df_consistencia["cons_lactating_cows_total_cattle"] = (
    preparar_regra_consistencia(
        df_consistencia["lactating_cows_total_cattle"].gt(15)
        & df_consistencia["lactating_cows_total_cattle"].lt(99)
    )
)


# ============================================================
# 7. PRODUTIVIDADE E MÃO DE OBRA
# ============================================================

# A produção diária por vaca em lactação deve ser maior que 3 e menor que 45 litros por vaca por dia.
df_consistencia["cons_milk_lactating_cow_day"] = (
    preparar_regra_consistencia(
        df_consistencia["milk_lactating_cow_day"].gt(3)
        & df_consistencia["milk_lactating_cow_day"].lt(45)
    )
)


# A produção diária por unidade de mão de obra deve ser maior que 20 e menor que 1.500 litros por trabalhador por dia.
df_consistencia["cons_milk_total_labor_day"] = (
    preparar_regra_consistencia(
        df_consistencia["milk_total_labor_day"].gt(20)
        & df_consistencia["milk_total_labor_day"].lt(1500)
    )
)


# O número de vacas em lactação por unidade de mão de obra deve ser positivo e menor que 70.
# O limite inferior positivo evita que propriedades com zero vacas em lactação sejam classificadas como consistentes.
df_consistencia["cons_lactating_cows_total_labor"] = (
    preparar_regra_consistencia(
        df_consistencia["lactating_cows_total_labor"].gt(0)
        & df_consistencia["lactating_cows_total_labor"].lt(70)
    )
)


# ============================================================
# 8. ALIMENTAÇÃO E CUSTOS
# ============================================================

# feeding_cost_milk_price está em percentual na base nova.
# O limite antigo de 0,15 a 1,50 corresponde a 15% a 150%.
df_consistencia["cons_feeding_cost_milk_price"] = (
    preparar_regra_consistencia(
        df_consistencia["feeding_cost_milk_price"].gt(15)
        & df_consistencia["feeding_cost_milk_price"].lt(150)
    )
)

# O custo do volumoso deve ser menor que R$ 3 por litro.
df_consistencia["cons_voluminous_cost_liter"] = (
    preparar_regra_consistencia(
        df_consistencia["voluminous_cost_liter"].lt(3)
    )
)

# O custo de concentrado e minerais deve ser maior que R$ 0,30 e menor que R$ 3,50 por litro.
df_consistencia["cons_concentrate_mineral_cost_liter"] = (
    preparar_regra_consistencia(
        df_consistencia["concentrate_mineral_cost_liter"].gt(0.3)
        & df_consistencia["concentrate_mineral_cost_liter"].lt(3.5)
    )
)

# O custo da mão de obra contratada deve ser menor que R$ 1 por litro.
df_consistencia["cons_hired_labor_cost_liter"] = (
    preparar_regra_consistencia(
        df_consistencia["hired_labor_cost_liter"].lt(1)
    )
)

# ============================================================
# 9. ESTOQUE DE CAPITAL
# ============================================================

# Verifica se o estoque total de capital por produção diária
# está abaixo do limite de consistência.
# df_consistencia["cons_total_capital_stock"] = (
#     preparar_regra_consistencia(
#         df_consistencia["total_capital_stock_milk_daily"].lt(30000)
#     )
# )

# ============================================================
# 10. MAPA DOS CRITÉRIOS
# ============================================================

# Relaciona cada coluna booleana ao nome que será exibido
# no relatório de critérios violados.
MAPA_CRITERIOS_CONSISTENCIA = {
    "cons_ccs": "CCS",
    "cons_cpp": "CPP",
    "cons_fat": "Gordura",
    "cons_protein": "Proteína",
    "cons_lactating_cows_total_cows": "VL/Total de vacas",
    "cons_lactating_cows_total_cattle": "VL/Rebanho total",
    "cons_milk_lactating_cow_day": "Produção/VL",
    "cons_milk_total_labor_day": "Produção/MDO",
    "cons_lactating_cows_total_labor": "VL/MDO",
    "cons_feeding_cost_milk_price": "Alimentação/Preço do leite",
    "cons_voluminous_cost_liter": "Custo de volumoso",
    "cons_concentrate_mineral_cost_liter": "Custo de concentrado",
    "cons_hired_labor_cost_liter": "Custo da MDO contratada",
    # "cons_total_capital_stock": "Estoque de capital fixo",
}


# Transforma as chaves do dicionário em uma lista.
# Essa lista contém todas as colunas de consistência.
colunas_criterios = list(
    MAPA_CRITERIOS_CONSISTENCIA.keys()
)


# ============================================================
# 11. CONTAR CRITÉRIOS ATENDIDOS E VIOLADOS
# ============================================================

# Soma os valores True de cada linha.
# No pandas, True equivale a 1 e False equivale a 0.
df_consistencia["total_consistency_criteria_ok"] = (
    df_consistencia[colunas_criterios]
    .sum(axis=1)
    .astype("int8")
)


# Salva a quantidade total de critérios avaliados.
df_consistencia["total_consistency_criteria"] = len(
    colunas_criterios
)


# Calcula quantos critérios não foram atendidos.
df_consistencia["total_consistency_criteria_violated"] = (
    df_consistencia["total_consistency_criteria"]
    - df_consistencia["total_consistency_criteria_ok"]
)


# ============================================================
# 12. CLASSIFICAÇÃO GERAL
# ============================================================

# Classifica como Consistente quando nenhum critério foi violado.
# Caso exista pelo menos uma violação, classifica como Inconsistente.
df_consistencia["consistency_status"] = np.where(
    df_consistencia["total_consistency_criteria_violated"].eq(0),
    "Consistente",
    "Inconsistente",
)


# Cria uma classificação numérica.
# Zero representa consistente e um representa inconsistente.
df_consistencia["consistency_id"] = np.where(
    df_consistencia["total_consistency_criteria_violated"].eq(0),
    0,
    1,
).astype("int8")


# ============================================================
# 13. LISTAR OS CRITÉRIOS VIOLADOS
# ============================================================

# Cria inicialmente uma coluna vazia.
df_consistencia["violated_consistency_criteria"] = ""


# Percorre cada critério e seu respectivo nome de exibição.
for coluna_criterio, nome_criterio in MAPA_CRITERIOS_CONSISTENCIA.items():

    # Identifica as linhas em que o critério não foi atendido.
    mascara_violacao = ~df_consistencia[coluna_criterio]

    # Adiciona o nome do critério à lista de violações da linha.
    df_consistencia.loc[
        mascara_violacao,
        "violated_consistency_criteria",
    ] += nome_criterio + "; "


# Remove o último ponto e vírgula e os espaços excedentes.
df_consistencia["violated_consistency_criteria"] = (
    df_consistencia["violated_consistency_criteria"]
    .str.rstrip("; ")
)


# Substitui textos vazios por "Nenhum".
# Isso acontece quando todos os critérios foram atendidos.
df_consistencia["violated_consistency_criteria"] = (
    df_consistencia["violated_consistency_criteria"]
    .replace("", "Nenhum")
)


# ============================================================
# 14. CONFERÊNCIA DO RESULTADO
# ============================================================

# Cria uma tabela resumida com a quantidade de registros
# consistentes e inconsistentes.
df_resumo_consistencia = (
    df_consistencia["consistency_status"]
    .value_counts(dropna=False)
    .rename_axis("consistency_status")
    .reset_index(name="records")
)


# Calcula o percentual de cada classificação.
df_resumo_consistencia["percentage"] = (
    df_resumo_consistencia["records"]
    .div(len(df_consistencia))
    .mul(100)
    .round(2)
)


# Exibe o resumo da classificação.
display(df_resumo_consistencia)


# Exibe uma amostra das principais colunas geradas.
display(
    df_consistencia[
        [
            "id_property",
            "reference_month",
            "consistency_status",
            "consistency_id",
            "total_consistency_criteria_ok",
            "total_consistency_criteria_violated",
            "violated_consistency_criteria",
        ]
    ]
    .head(20)
)

In [ ]:
df_consistencia.columns.tolist() 

### Preparar os dados para o Excel

In [ ]:
def preparar_dataframe_para_excel(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepara um DataFrame para exportação pelo openpyxl.
    """

    df_excel = df.copy()

    # Excel não trabalha com infinito
    df_excel = df_excel.replace([np.inf, -np.inf], np.nan)

    # Excel não aceita datas com timezone
    for coluna in df_excel.columns:
        if isinstance( df_excel[coluna].dtype, pd.DatetimeTZDtype, ):
            df_excel[coluna] = ( df_excel[coluna] .dt.tz_localize(None) )
    
    return df_excel

#### Exportar uma planilha por view

In [ ]:
# ============================================================
# EXPORTAR DF_INTEGRADA PARA EXCEL
# ============================================================

DATA_EXPORTACAO = datetime.now().strftime("%Y_%m_%d")
PASTA_SAIDA = (Path.cwd().parent / "data" / "outputs" / "monthly")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
CAMINHO_ARQUIVO = (PASTA_SAIDA / f"{DATA_EXPORTACAO}_indicadores_mensais.xlsx")

# Criar uma cópia para não alterar a tabela original
df_exportacao = df_consistencia.copy()

# Remover valores infinitos
df_exportacao = df_exportacao.replace([np.inf, -np.inf], np.nan)

# Ordenar a tabela
df_exportacao = (df_exportacao.sort_values( ["id_property", "reference_month"] ) .reset_index(drop=True))

# Validar duplicidades
if df_exportacao.duplicated(["id_property", "reference_month"]).any():
    raise ValueError("Existem duplicidades por id_property e reference_month.")

# Preparar os dados para o Excel
df_exportacao = preparar_dataframe_para_excel(df_exportacao)

# Exportar com a formatação padrão
exportar_varias_abas_xlsx(abas={"Indicadores Mensais": df_exportacao }, caminho_saida=CAMINHO_ARQUIVO, fonte="Aptos")

print("Exportação concluída com sucesso.")
print(f"Linhas exportadas: {len(df_exportacao):,}")
print(f"Arquivo: {CAMINHO_ARQUIVO}")

## Indicadores Anuais

#### Janela Móvel de 12 meses completo

In [ ]:
# ============================================================
# INDICADORES ANUAIS: JANELA MÓVEL DE 12 MESES COMPLETOS
# ============================================================
df_mensal_anuais = df_consistencia.copy()
df_mensal_anuais["reference_month"] = (pd.to_datetime(df_mensal_anuais["reference_month"], errors="coerce").dt.to_period("M").dt.to_timestamp())
df_mensal_anuais = (df_mensal_anuais.dropna(subset=["id_property", "reference_month"]).sort_values(["id_property", "reference_month"]).drop_duplicates(["id_property", "reference_month"], keep="last").reset_index(drop=True))

# Criar coluna de CCS, CPP, Gordura, Proteína ainda não calculados
df_mensal_anuais['abs_ccs'] = df_mensal_anuais['ccs'] * df_mensal_anuais['milk_produced']
df_mensal_anuais['abs_cpp'] = df_mensal_anuais['cpp'] * df_mensal_anuais['milk_produced']
df_mensal_anuais['abs_fat'] = df_mensal_anuais['fat'] * df_mensal_anuais['milk_produced']
df_mensal_anuais['abs_protein'] = df_mensal_anuais['protein'] * df_mensal_anuais['milk_produced']

# ============================================================
# ÁREA: RECONSTRUIR "COM RESERVA" E "FORRAGEIRA" (ainda não calculadas na Feature Engineering)
# ============================================================
# hectares_atividade já é a área "sem reserva" (benfeitorias/estradas + forrageiras, próprio + arrendado).
# Somando de volta a Reserva Legal e APP chegamos na área "com reserva" (equivalente à antiga areaAtividadeReserva).
COLUNAS_RESERVA_LEGAL = [c for c in ["hectares_owned_app_reserva_legal", "hectares_rented_app_reserva_legal"] if c in df_mensal_anuais.columns]
if COLUNAS_RESERVA_LEGAL:
    df_mensal_anuais["hectares_atividade_com_reserva"] = (
        df_mensal_anuais["hectares_atividade"].fillna(0)
        + df_mensal_anuais[COLUNAS_RESERVA_LEGAL].sum(axis=1, skipna=True)
    )

# Área de forrageira (própria + arrendada), equivalente à antiga areaForrageira.
COLUNAS_FORRAGEIRA = [c for c in ["hectares_owned_forrageiras", "hectares_rented_forrageiras"] if c in df_mensal_anuais.columns]
if COLUNAS_FORRAGEIRA:
    df_mensal_anuais["hectares_forrageira"] = df_mensal_anuais[COLUNAS_FORRAGEIRA].sum(axis=1, skipna=True)

COLUNAS_SOMA_ANUAL = [
    # === Produção de Leite === #
    "milk_produced", "milk_volume_sold", "milk_volume_derivatives", "discarded_quantity",
    # === Renda === #
    "total_milk_revenue", "total_activity_revenue", "concentrated_sold", "voluminous_sold",
    # === COE === #
    "general_expenses", "administration", "land_lease", "technical_assistance",
    "repairs", "hormones", "taxes_fees", "medicines_vaccines", "bedding_replacement",
    "reproduction", "milk_replacer", "milking_material", "milk_calves", "energy", "fuel",
    "voluminous_amount_total", "concentrate_amount_total", "mineral_amount_total",
    "hired_labor_expenses", "other_operating_expenses",
    "feeding_cost", "concentrate_mineral_cost",
    # === COT === #
    "family_labor_expenses", "monthly_depreciation_benfeitorias", "monthly_depreciation_maquinas_e_equipamentos",
    # === Alimentação (quantidades consumidas) === #
    "concentrate_consumed_quantity",
    "voluminous_consumed_quantity",
    "mineral_consumed_quantity",
    # === MDO === #
    "family_labor_quantity",
    "hired_labor_quantity",
    # === QUALIDADE (auxiliares para média ponderada) === #
    "abs_ccs", "abs_cpp", "abs_fat", "abs_protein",
    # === Estoque de Capital === #
    'monthly_depreciation_benfeitorias',
    'monthly_depreciation_maquinas_e_equipamentos',
    'monthly_average_capital_stock_benfeitorias',
    'monthly_average_capital_stock_maquinas_e_equipamentos'
]
COLUNAS_MEDIA_ANUAL = [
    # === Animais === #
    "lactating_cows", "total_cows", "total_cattle",
    # === Quantidade de MDO === #
    "hired_labor_quantity", "family_labor_quantity", "total_labor_quantity",
    # === Área === #
    "hectares_atividade", "hectares_atividade_com_reserva", "hectares_forrageira",
    "hectares_arrendada", "hectares_propria", "hectares_total",
    "raw_land_value_medio_ponderado",
    # === Estoque de Capital (é um estoque médio, não um fluxo: usar média, não soma) === #
    "animal_capital_stock", "land_capital_stock"
]
COLUNAS_QUALIDADE = ["ccs", "cpp", "fat", "protein"]
COLUNAS_SOMA_ANUAL = [c for c in COLUNAS_SOMA_ANUAL if c in df_mensal_anuais.columns]
COLUNAS_MEDIA_ANUAL = [c for c in COLUNAS_MEDIA_ANUAL if c in df_mensal_anuais.columns]
COLUNAS_QUALIDADE = [c for c in COLUNAS_QUALIDADE if c in df_mensal_anuais.columns]

# ============================================================
# COBERTURA DE LANÇAMENTO DE DADOS POR JANELA ANUAL
# ============================================================
# A continuidade da janela (12 meses seguidos) só é garantida para a vw_revenue,
# que é a base de df_mensal_anuais. Isso não impede que uma propriedade tenha
# lançado receita todo mês mas deixado de lançar despesa/mão de obra/alimentação
# em alguns meses daquela mesma janela — o que subestima COE/COT/CT sem gerar
# nenhum erro (a soma anual simplesmente usa os meses que existem).
# Este relatório mede, para cada janela, quantos dos 12 meses têm cada fonte.
MAPA_COBERTURA = {
    "has_revenue_data": "months_with_revenue_data",
    "has_cattle_data": "months_with_cattle_data",
    "has_expense_data": "months_with_expense_data",
    "has_feeding_data": "months_with_feeding_data",
    "has_labor_data": "months_with_labor_data",
    "has_own_milk_data": "months_with_own_milk_data",
    "has_asset_data": "months_with_asset_data",
    "has_active_area_month": "months_with_active_area_data",
    "has_dairy_production_system_data": "months_with_dairy_production_system_data",
}
MAPA_COBERTURA = {k: v for k, v in MAPA_COBERTURA.items() if k in df_mensal_anuais.columns}

# Fontes usadas diretamente no cálculo de COE/COT/CT: são as que mais importam
# para explicar uma possível subestimação de custo.
FONTES_CRITICAS_PARA_CUSTO = {
    "has_expense_data": "Despesas (vw_expense)",
    "has_feeding_data": "Alimentação (vw_feeding)",
    "has_labor_data": "Mão de obra (vw_labor)",
    "has_asset_data": "Patrimônio/Depreciação (mvw_asset_payment_history)",
}

registros_anuais = []
registros_cobertura = []
for id_property, grupo in df_mensal_anuais.groupby("id_property", sort=False):
    grupo = grupo.sort_values("reference_month").reset_index(drop=True)
    for fim in range(11, len(grupo)):
        janela = grupo.iloc[fim - 11:fim + 1]
        meses = janela["reference_month"].dt.to_period("M")
        esperado = pd.period_range(meses.iloc[0], meses.iloc[-1], freq="M")
        if len(esperado) != 12 or list(meses) != list(esperado):
            continue
        registro = janela.iloc[-1].to_dict()
        registro["annual_period_start"] = janela["reference_month"].iloc[0]
        registro["annual_period_end"] = janela["reference_month"].iloc[-1]
        registro["annual_period"] = f"{registro['annual_period_start']:%b/%y}-{registro['annual_period_end']:%b/%y}"
        registro["months_in_annual_window"] = 12
        for coluna in COLUNAS_SOMA_ANUAL:
            registro[f"{coluna}_annual"] = janela[coluna].sum(min_count=1)
        for coluna in COLUNAS_MEDIA_ANUAL:
            registro[f"{coluna}_annual_average"] = janela[coluna].mean()
        volume = pd.to_numeric(janela["milk_produced"], errors="coerce")
        for coluna in COLUNAS_QUALIDADE:
            valores = pd.to_numeric(janela[coluna], errors="coerce")
            validos = valores.notna() & volume.notna() & volume.gt(0)
            registro[f"{coluna}_annual_weighted_average"] = (np.average(valores[validos], weights=volume[validos]) if validos.any() else np.nan)
        inconsistencias = pd.to_numeric(janela["consistency_id"], errors="coerce")
        registro["inconsistent_months_annual"] = inconsistencias.sum(min_count=1)
        registro["annual_consistency_id"] = int(not inconsistencias.eq(0).all())
        registro["annual_consistency_status"] = "Consistente" if registro["annual_consistency_id"] == 0 else "Inconsistente"
        registros_anuais.append(registro)

        # --- Cobertura de lançamento de dados desta mesma janela (relatório separado) ---
        registro_cobertura = {
            "id_property": id_property,
            "annual_period_start": registro["annual_period_start"],
            "annual_period_end": registro["annual_period_end"],
            "annual_period": registro["annual_period"],
        }
        fontes_incompletas = []
        for coluna_flag, nome_coluna_cobertura in MAPA_COBERTURA.items():
            meses_com_dado = int(janela[coluna_flag].sum())
            registro_cobertura[nome_coluna_cobertura] = meses_com_dado
            if coluna_flag in FONTES_CRITICAS_PARA_CUSTO and meses_com_dado < 12:
                fontes_incompletas.append(f"{FONTES_CRITICAS_PARA_CUSTO[coluna_flag]}: {meses_com_dado}/12")
        registro_cobertura["cost_data_status"] = "Completa" if not fontes_incompletas else "Incompleta"
        registro_cobertura["cost_data_gaps"] = "; ".join(fontes_incompletas) if fontes_incompletas else "Nenhum"
        registros_cobertura.append(registro_cobertura)

df_anuais = pd.DataFrame(registros_anuais)
if df_anuais.empty:
    raise ValueError("Nenhuma propriedade possui uma janela completa de 12 meses consecutivos.")
if df_anuais.duplicated(["id_property", "annual_period_end"]).any():
    raise ValueError("Foram geradas janelas anuais duplicadas por propriedade e mês final.")
print(f"Janelas anuais calculadas: {len(df_anuais):,}")

# ============================================================
# RELATÓRIO DE COBERTURA DE DADOS (SEPARADO DOS INDICADORES ANUAIS)
# ============================================================
# Este relatório não entra em df_anuais / df_indicadores_anuais / df_calculo_medias.
# Ele existe para diagnosticar, por propriedade e janela anual, se algum mês deixou
# de ter dado lançado em despesas/alimentação/mão de obra/patrimônio — o que
# subestimaria COE, COT e CT silenciosamente sem isso ficar visível em nenhum outro lugar.
df_cobertura_anual = pd.DataFrame(registros_cobertura)
df_cobertura_anual = df_cobertura_anual.merge(
    df_anuais[["id_property", "annual_period_end", "property_name", "labor_rural_code"]],
    on=["id_property", "annual_period_end"],
    how="left",
) if {"property_name", "labor_rural_code"}.issubset(df_anuais.columns) else df_cobertura_anual
df_cobertura_anual = df_cobertura_anual.sort_values(["id_property", "annual_period_end"]).reset_index(drop=True)

janelas_incompletas = int((df_cobertura_anual["cost_data_status"] == "Incompleta").sum())
print(f"Janelas anuais com lançamento de custo incompleto: {janelas_incompletas:,} de {len(df_cobertura_anual):,}")
display(df_cobertura_anual.head())

In [ ]:
df_anuais.columns.to_list()

#### Indicadores Derivados da Janela Anual

In [ ]:
pd.set_option('future.no_silent_downcasting', True)
def dividir_seguro(numerador: pd.Series, denominador: pd.Series) -> pd.Series:
    denom_seguro = denominador.where(denominador != 0, np.nan)
    resultado = numerador / denom_seguro
    return resultado.where(resultado.abs() != np.inf, np.nan)

def coluna_ou_none(df: pd.DataFrame, nome: str) -> pd.Series:
    """Retorna a coluna se ela existir; caso contrário, uma série de None (mesmo índice)."""
    if nome in df.columns:
        return df[nome]
    return pd.Series([None] * len(df), index=df.index)

dias_ano = (df_anuais["annual_period_end"] + pd.offsets.MonthEnd(0) - df_anuais["annual_period_start"] + pd.Timedelta(days=1)).dt.days


# ============================================================
# 1. RENDA
# ============================================================
df_anuais['rbl_rba'] = dividir_seguro(df_anuais['total_milk_revenue_annual'], df_anuais['total_activity_revenue_annual'])
# Versão em percentual (0-100), equivalente ao antigo rbl_rba já multiplicado por 100.
df_anuais['rbl_rba_percentage_annual'] = df_anuais['rbl_rba'] * 100

# ============================================================
# 1. PRODUÇÃO, QUALIDADE E MÃO DE OBRA (já existentes, mantidos)
# ============================================================
df_anuais["milk_daily_annual"] = dividir_seguro(df_anuais["milk_produced_annual"], dias_ano)
df_anuais["milk_revenue_liter_annual"] = dividir_seguro(df_anuais["total_milk_revenue_annual"], df_anuais["milk_produced_annual"])
df_anuais["milk_lactating_cow_day_annual"] = dividir_seguro(df_anuais["milk_daily_annual"], df_anuais["lactating_cows_annual_average"])
df_anuais["milk_daily_total_cows_annual"] = dividir_seguro(df_anuais["milk_daily_annual"], df_anuais["total_cows_annual_average"])
df_anuais["lactating_cows_total_cows_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], df_anuais["total_cows_annual_average"]) * 100
df_anuais["lactating_cows_total_cattle_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], df_anuais["total_cattle_annual_average"]) * 100
df_anuais["milk_total_labor_day_annual"] = dividir_seguro(df_anuais["milk_daily_annual"], df_anuais["total_labor_quantity_annual_average"])
df_anuais["lactating_cows_total_labor_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], df_anuais["total_labor_quantity_annual_average"])

# ============================================================
# 2. ÁREA (com reserva, forrageira e produção por área)
# ============================================================
df_anuais["milk_hectare_activity_annual"] = dividir_seguro(df_anuais["milk_produced_annual"], df_anuais["hectares_atividade_annual_average"])
df_anuais["milk_hectare_activity_com_reserva_annual"] = dividir_seguro(df_anuais["milk_produced_annual"], coluna_ou_none(df_anuais, "hectares_atividade_com_reserva_annual_average"))
df_anuais["milk_hectare_forrageira_annual"] = dividir_seguro(df_anuais["milk_produced_annual"], coluna_ou_none(df_anuais, "hectares_forrageira_annual_average"))
df_anuais["lactating_cows_hectare_activity_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], df_anuais["hectares_atividade_annual_average"])
df_anuais["lactating_cows_hectare_activity_com_reserva_annual"] = dividir_seguro(df_anuais["lactating_cows_annual_average"], coluna_ou_none(df_anuais, "hectares_atividade_com_reserva_annual_average"))
df_anuais["rented_area_percentage_annual"] = dividir_seguro(df_anuais["hectares_arrendada_annual_average"], df_anuais["hectares_total_annual_average"]) * 100
df_anuais["land_value_per_hectare_annual"] = coluna_ou_none(df_anuais, "raw_land_value_medio_ponderado_annual_average")

# ============================================================
# 3. QUALIDADE DO LEITE: GORDURA/PROTEÍNA POR VACA EM LACTAÇÃO/DIA
# ============================================================
# Réplica do indicador antigo Gordura_VL / Proteina_VL:
# (produção anual/365) * 1,032 * (percentual médio ponderado/100) / vacas em lactação médias
df_anuais["fat_lactating_cow_day_annual"] = dividir_seguro(
    (df_anuais["milk_produced_annual"] / 365) * 1.032 * (coluna_ou_none(df_anuais, "fat_annual_weighted_average") / 100),
    df_anuais["lactating_cows_annual_average"],
)
df_anuais["protein_lactating_cow_day_annual"] = dividir_seguro(
    (df_anuais["milk_produced_annual"] / 365) * 1.032 * (coluna_ou_none(df_anuais, "protein_annual_weighted_average") / 100),
    df_anuais["lactating_cows_annual_average"],
)

# ============================================================
# 4. ALIMENTAÇÃO: CUSTO, PREÇO DO CONCENTRADO E RELAÇÃO DE TROCA
# ============================================================
df_anuais["feeding_cost_liter_annual"] = dividir_seguro(df_anuais["feeding_cost_annual"], df_anuais["milk_produced_annual"])
df_anuais["concentrate_mineral_cost_liter_annual"] = dividir_seguro(df_anuais["concentrate_mineral_cost_annual"], df_anuais["milk_produced_annual"])
df_anuais["feeding_cost_milk_price_annual"] = dividir_seguro(df_anuais["feeding_cost_liter_annual"], df_anuais["milk_revenue_liter_annual"]) * 100

df_anuais["concentrate_mineral_consumed_quantity_annual"] = (
    df_anuais[["concentrate_consumed_quantity_annual", "mineral_consumed_quantity_annual"]].sum(axis=1, min_count=1)
    if {"concentrate_consumed_quantity_annual", "mineral_consumed_quantity_annual"}.issubset(df_anuais.columns)
    else None
)
df_anuais["concentrate_mineral_price_annual"] = dividir_seguro(df_anuais["concentrate_mineral_cost_annual"], coluna_ou_none(df_anuais, "concentrate_mineral_consumed_quantity_annual"))
df_anuais["milk_concentrate_exchange_ratio_annual"] = dividir_seguro(df_anuais["milk_revenue_liter_annual"], coluna_ou_none(df_anuais, "concentrate_mineral_price_annual"))

# ============================================================
# 4b. AGREGAÇÕES DE CUSTO USADAS NO CÁLCULO DE MÉDIAS ANTIGO
# ============================================================
# Aleitamento (sucedâneo comprado + leite de bezerro), equivalente ao antigo def_aleitaento_somaMovel.
COMPONENTES_ALEITAMENTO = [c for c in ["milk_replacer_annual", "milk_calves_annual"] if c in df_anuais.columns]
df_anuais["aleitamento_annual"] = df_anuais[COMPONENTES_ALEITAMENTO].sum(axis=1, min_count=1) if COMPONENTES_ALEITAMENTO else None

# Energia + combustível, equivalente ao antigo custoEnergiaCombustivel.
COMPONENTES_ENERGIA_COMBUSTIVEL = [c for c in ["energy_annual", "fuel_annual"] if c in df_anuais.columns]
df_anuais["energy_fuel_cost_annual"] = df_anuais[COMPONENTES_ENERGIA_COMBUSTIVEL].sum(axis=1, min_count=1) if COMPONENTES_ENERGIA_COMBUSTIVEL else None

# Depreciação total do estoque de capital (benfeitorias + máquinas), equivalente ao antigo
# depreciacaoEstoqueCapital_somaMovel. Falta o componente de forrageiras não-anuais/plantio.
COMPONENTES_DEPRECIACAO_TOTAL = [c for c in ["monthly_depreciation_benfeitorias_annual", "monthly_depreciation_maquinas_e_equipamentos_annual"] if c in df_anuais.columns]
df_anuais["total_depreciation_annual"] = df_anuais[COMPONENTES_DEPRECIACAO_TOTAL].sum(axis=1, min_count=1) if COMPONENTES_DEPRECIACAO_TOTAL else None

# ============================================================
# 5. COE, COT E CT (CUSTO OPERACIONAL EFETIVO, TOTAL E CUSTO TOTAL)
# ============================================================
COMPONENTES_COE_ATIVIDADE_ANUAL = [
    # "other_operating_expenses_annual" foi removida daqui de propósito: ela já é a soma de
    # general_expenses, administration, land_lease, technical_assistance, repairs, hormones,
    # taxes_fees, medicines_vaccines, bedding_replacement, reproduction e milk_replacer
    # (ver COLUNAS_OUTRAS_DESPESAS na Feature Engineering). Mantê-la aqui junto com essas
    # mesmas colunas individuais contava esse bloco de despesas duas vezes no COE.
    "general_expenses_annual", "administration_annual", "land_lease_annual", "technical_assistance_annual",
    "repairs_annual", "hormones_annual", "taxes_fees_annual", "medicines_vaccines_annual", "bedding_replacement_annual",
    "reproduction_annual", "milk_replacer_annual", "milk_calves_annual", "energy_annual", "fuel_annual",
    "voluminous_amount_total_annual", "concentrate_amount_total_annual", "mineral_amount_total_annual",
    "hired_labor_expenses_annual"
]
COMPONENTES_COE_LEITE = [
    # COE do Leite é o COE da Atividade vezes a renda bruta do leite/renda bruta da atividade + Material de Ordenha e sem custos com aleitamento.
    # (mesmo ajuste acima: "other_operating_expenses_annual" removida para não contar essas despesas duas vezes.)
    "general_expenses_annual", "administration_annual", "land_lease_annual", "technical_assistance_annual",
    "repairs_annual", "hormones_annual", "taxes_fees_annual", "medicines_vaccines_annual", "bedding_replacement_annual",
    "reproduction_annual", "milk_replacer_annual", "energy_annual", "fuel_annual", # Removido aleitamento daqui
    "voluminous_amount_total_annual", "concentrate_amount_total_annual", "mineral_amount_total_annual",
    "hired_labor_expenses_annual"
] # Isso aqui será multiplicado por Renda do Leite / Renda da Atividade.

# COE da Atividade
COMPONENTES_COE_ATIVIDADE_ANUAL = [c for c in COMPONENTES_COE_ATIVIDADE_ANUAL if c in df_anuais.columns]
df_anuais["coe_activity_annual"] = df_anuais[COMPONENTES_COE_ATIVIDADE_ANUAL].sum(axis=1, min_count=1) if COMPONENTES_COE_ATIVIDADE_ANUAL else None

# COE do Leite
COMPONENTES_COE_LEITE = [c for c in COMPONENTES_COE_LEITE if c in df_anuais.columns]
df_anuais["coe_milk_annual"] = (
    (df_anuais[COMPONENTES_COE_LEITE].sum(axis=1, min_count=1)) * df_anuais["rbl_rba"]
    + coluna_ou_none(df_anuais, "milking_material_annual")
) if COMPONENTES_COE_LEITE else None

COMPONENTES_COT_ADICIONAIS = [
    "family_labor_expenses_annual",
    "monthly_depreciation_benfeitorias_annual",
    "monthly_depreciation_maquinas_e_equipamentos_annual",
    # Falta o componente de depreciação de forrageiras não-anuais/plantio: sem fonte de dados ainda -> None.
]

COMPONENTES_COT_ADICIONAIS = [c for c in COMPONENTES_COT_ADICIONAIS if c in df_anuais.columns]
if "coe_activity_annual" in df_anuais.columns and COMPONENTES_COT_ADICIONAIS:
    df_anuais["cot_annual"] = df_anuais[["coe_activity_annual"] + COMPONENTES_COT_ADICIONAIS].sum(axis=1, min_count=1)
else:
    df_anuais["cot_annual"] = None

df_anuais['total_capital_stock_annual_average'] = (
    df_anuais[
        [
            'monthly_average_capital_stock_benfeitorias_annual',
            'monthly_average_capital_stock_maquinas_e_equipamentos_annual',
            'animal_capital_stock_annual_average',
            'land_capital_stock_annual_average'
        ]
    ]
    .sum(axis=1)
)
# Estoque de capital sem terra (para o custo de oportunidade do capital).
if {"total_capital_stock_annual_average", "land_capital_stock_annual_average"}.issubset(df_anuais.columns):
    df_anuais["capital_stock_no_land_annual_average"] = (
        df_anuais["total_capital_stock_annual_average"] - df_anuais["land_capital_stock_annual_average"]
    )
else:
    df_anuais["capital_stock_no_land_annual_average"] = None

# Custo de oportunidade do capital: 6% ao ano sobre o estoque de capital sem terra.
df_anuais["capital_opportunity_cost_annual"] = coluna_ou_none(df_anuais, "capital_stock_no_land_annual_average") * 0.06

if df_anuais["cot_annual"].notna().any() and df_anuais["capital_opportunity_cost_annual"].notna().any():
    df_anuais["ct_annual"] = df_anuais[["cot_annual", "capital_opportunity_cost_annual"]].sum(axis=1, min_count=1)
else:
    df_anuais["ct_annual"] = None

# ============================================================
# 6. RESULTADO ECONÔMICO: MARGENS, LUCRO E RCMA
# ============================================================
df_anuais["gross_margin_annual"] = df_anuais["total_activity_revenue_annual"] - coluna_ou_none(df_anuais, "coe_activity_annual")
df_anuais["net_margin_annual"] = df_anuais["total_activity_revenue_annual"] - coluna_ou_none(df_anuais, "cot_annual")
df_anuais["profit_annual"] = df_anuais["total_activity_revenue_annual"] - coluna_ou_none(df_anuais, "ct_annual")

COMPONENTES_RCMA = [c for c in ["concentrate_amount_total_annual", "mineral_amount_total_annual", "voluminous_amount_total_annual"] if c in df_anuais.columns]
if COMPONENTES_RCMA:
    df_anuais["rcma_annual"] = df_anuais["total_activity_revenue_annual"] - df_anuais[COMPONENTES_RCMA].sum(axis=1, min_count=1)
else:
    df_anuais["rcma_annual"] = None
df_anuais["rcma_lactating_cow_day_annual"] = dividir_seguro(coluna_ou_none(df_anuais, "rcma_annual"), df_anuais["lactating_cows_annual_average"] * 365)

# ============================================================
# 7. INDICADORES DE EFICIÊNCIA E RETORNO SOBRE O CAPITAL
# ============================================================
df_anuais["turnover_rate_annual"] = dividir_seguro(df_anuais["total_activity_revenue_annual"], df_anuais["total_capital_stock_annual_average"]) * 100
df_anuais["profitability_annual"] = dividir_seguro(coluna_ou_none(df_anuais, "net_margin_annual"), df_anuais["total_activity_revenue_annual"]) * 100

taxa_retorno_sem_terra = dividir_seguro(coluna_ou_none(df_anuais, "net_margin_annual"), coluna_ou_none(df_anuais, "capital_stock_no_land_annual_average")) * 100
df_anuais["capital_return_rate_no_land_annual"] = taxa_retorno_sem_terra.clip(lower=0)

taxa_retorno_com_terra = dividir_seguro(coluna_ou_none(df_anuais, "net_margin_annual"), df_anuais["total_capital_stock_annual_average"]) * 100
df_anuais["capital_return_rate_annual"] = taxa_retorno_com_terra.clip(lower=0)
# Versão sem o piso em zero, usada como auxiliar de estratificação (equivalente à antiga taxaRetornoCapitalComTerra_Geral).
df_anuais["capital_return_rate_annual_unfiltered"] = taxa_retorno_com_terra

# Pontos de cobertura (operacional e total), em litros/dia equivalentes.
df_anuais["operating_coverage_point_annual"] = dividir_seguro(coluna_ou_none(df_anuais, "cot_annual") / (12 * 30.42), df_anuais["milk_revenue_liter_annual"])
df_anuais["total_coverage_point_annual"] = dividir_seguro(coluna_ou_none(df_anuais, "ct_annual") / (12 * 30.42), df_anuais["milk_revenue_liter_annual"])

# ============================================================
df_anuais = df_anuais.copy()  # Defragmenta antes do bloco extensivo de colunas
# 8. RB, COE, COT, CT, MARGENS E LUCRO SOB DIFERENTES DENOMINADORES
# ============================================================
# Réplica sistemática do antigo bloco "colunas_economicas": cada medida econômica anual
# expressa por litro, % da receita, litros equivalentes, por vaca em lactação, por total
# de vacas e por hectare de área da atividade (sem e com reserva).
MEDIDAS_ECONOMICAS_ANUAIS = {
    "total_activity_revenue_annual": "activity_revenue",
    "coe_activity_annual": "coe",
    "cot_annual": "cot",
    "ct_annual": "ct",
    "gross_margin_annual": "gross_margin",
    "net_margin_annual": "net_margin",
    "profit_annual": "profit",
}
for coluna_base, prefixo in MEDIDAS_ECONOMICAS_ANUAIS.items():
    valores_base = coluna_ou_none(df_anuais, coluna_base)
    df_anuais[f"{prefixo}_liter_annual"] = dividir_seguro(valores_base, df_anuais["milk_produced_annual"])
    df_anuais[f"{prefixo}_milk_revenue_percentage_annual"] = dividir_seguro(valores_base, df_anuais["total_activity_revenue_annual"]) * 100
    df_anuais[f"{prefixo}_milk_equivalent_liters_annual"] = dividir_seguro(valores_base, df_anuais["milk_revenue_liter_annual"])
    df_anuais[f"{prefixo}_lactating_cow_annual"] = dividir_seguro(valores_base, df_anuais["lactating_cows_annual_average"])
    df_anuais[f"{prefixo}_total_cows_annual"] = dividir_seguro(valores_base, df_anuais["total_cows_annual_average"])
    df_anuais[f"{prefixo}_hectare_activity_annual"] = dividir_seguro(valores_base, df_anuais["hectares_atividade_annual_average"])
    df_anuais[f"{prefixo}_hectare_activity_com_reserva_annual"] = dividir_seguro(valores_base, coluna_ou_none(df_anuais, "hectares_atividade_com_reserva_annual_average"))

# ============================================================
# 9. ESTOQUE DE CAPITAL: TOTAL E QUEBRA POR COMPONENTE
# ============================================================
df_anuais["total_capital_stock_milk_daily_annual"] = dividir_seguro(df_anuais["total_capital_stock_annual_average"], df_anuais["milk_daily_annual"])
df_anuais["total_capital_stock_lactating_cow_annual"] = dividir_seguro(df_anuais["total_capital_stock_annual_average"], df_anuais["lactating_cows_annual_average"])

COMPONENTES_ESTOQUE_CAPITAL = {
    "monthly_average_capital_stock_benfeitorias_annual_average": "benfeitorias",
    "monthly_average_capital_stock_maquinas_e_equipamentos_annual_average": "maquinas_e_equipamentos",
    "animal_capital_stock_annual_average": "animais",
    "land_capital_stock_annual_average": "terra",
}
for coluna_base, sufixo in COMPONENTES_ESTOQUE_CAPITAL.items():
    df_anuais[f"capital_stock_{sufixo}_share_annual"] = dividir_seguro(coluna_ou_none(df_anuais, coluna_base), df_anuais["total_capital_stock_annual_average"]) * 100
# Estoque de capital em forrageiras não-anuais/plantio: sem fonte de dados ainda -> None.
df_anuais["capital_stock_forrageira_annual_average"] = None
df_anuais["capital_stock_forrageira_share_annual"] = None
df_anuais["forrageira_depreciation_annual"] = None

# ============================================================
df_anuais = df_anuais.copy()  # Defragmenta antes de adicionar placeholders
# 10. INVESTIMENTO: SEM FONTE DE DADOS AINDA (colunas mantidas como None)
# ============================================================
for coluna in [
    "investment_animals_annual",
    "investment_machinery_annual",
    "investment_improvements_annual",
    "investment_land_annual",
    "investment_total_annual",
    "investment_animals_capital_stock_annual",
    "investment_machinery_capital_stock_annual",
    "investment_improvements_capital_stock_annual",
    "investment_total_capital_stock_annual",
    "investment_revenue_annual",
    "investment_gross_margin_annual",
]:
    df_anuais[coluna] = None

# ============================================================
# 11. DIMENSÃO E CAMPOS AUXILIARES SEM FONTE DE DADOS NO MODELO NOVO
# ============================================================
df_anuais["agroindustry_code"] = None     # Código da agroindústria: só o nome está disponível.
df_anuais["production_system_annual"] = coluna_ou_none(df_anuais, "production_system")

# Placeholders herdados do notebook antigo (também eram sempre vazios ou não-informativos lá).
# "filter_1" NAO entra aqui: e calculado abaixo, a partir do merge com o consultor.
for coluna in ["filter_2", "filter_3", "filter_4", "monthly_status_annual", "lab_tests", "milk_quality_notes", "outsourcing", "transport", "vaccines"]:
    df_anuais[coluna] = None

# ============================================================
# CONSULTOR: MERGE TARDIO (DE PROPOSITO) E FILTRO 1
# Regra de negócio do Cálculo de Médias:
# - a propriedade continua sendo calculada uma única vez;
# - na análise de médias, a mesma propriedade pode ser duplicada para cada consultor vinculado;
# - essa duplicação é intencional e serve para análise por consultor;
# - a base duplicada não deve ser usada para consolidar totais gerais sem reagrupamento por propriedade.
# ============================================================
# Feito aqui, sobre df_anuais ja agregado -- e nao em df_integrada -- porque um
# merge cedo duplicaria linhas mensais para fazendas com mais de um consultor
# e inflaria todas as somas anuais pelo numero de consultores. Aqui a janela ja
# esta fechada e somada; duplicar a linha so repete o resultado, sem somar de novo.
if "df_dim_consultor" in dir() and not df_dim_consultor.empty:
    df_anuais = df_anuais.drop(columns=["id_consultor", "nome_consultor"], errors="ignore")
    df_anuais = df_anuais.merge(
        df_dim_consultor[["id_property", "id_consultor", "nome_consultor"]],
        on="id_property",
        how="left",
    )
    df_anuais["consultant_name"] = df_anuais["nome_consultor"]
    
    # Filtro 1: quantos consultores distintos (= quantas vezes esta linha fazenda-periodo se repete).
    df_anuais["filter_1"] = (
        df_anuais.groupby(["id_property", "annual_period"])["id_consultor"]
        .transform("nunique")
    )
else:
    df_anuais["consultant_name"] = None
    df_anuais["filter_1"] = None

# Desfragmenta o DataFrame após todas as inserções de colunas (resolve PerformanceWarning).
df_anuais = df_anuais.copy()
df_anuais = df_anuais.replace([np.inf, -np.inf], np.nan)

# ============================================================
# 12. MONTAR A TABELA FINAL
# ============================================================
COLUNAS_RESULTADO_ANUAL = [
    # --- Identificação e período --- #
    "id_property", "annual_period_start", "annual_period_end", "annual_period",
    "annual_consistency_status", "annual_consistency_id", "inconsistent_months_annual",
    "production_system_annual",
    # --- Produção, qualidade e mão de obra --- #
    "milk_produced_annual", "milk_daily_annual", "milk_revenue_liter_annual",
    "milk_lactating_cow_day_annual", "lactating_cows_total_cows_annual",
    "lactating_cows_total_cattle_annual", "milk_total_labor_day_annual",
    "lactating_cows_total_labor_annual", "fat_lactating_cow_day_annual", "protein_lactating_cow_day_annual",
    "milk_daily_total_cows_annual", "rbl_rba_percentage_annual",
    # --- Área --- #
    "milk_hectare_activity_annual", "milk_hectare_activity_com_reserva_annual", "milk_hectare_forrageira_annual",
    "lactating_cows_hectare_activity_annual", "lactating_cows_hectare_activity_com_reserva_annual",
    "rented_area_percentage_annual", "land_value_per_hectare_annual",
    # --- Alimentação --- #
    "feeding_cost_liter_annual", "concentrate_mineral_cost_liter_annual", "feeding_cost_milk_price_annual",
    "concentrate_mineral_price_annual", "milk_concentrate_exchange_ratio_annual",
    # --- Custos agregados --- #
    "coe_activity_annual", "coe_milk_annual", "cot_annual", "ct_annual", "capital_opportunity_cost_annual",
    "aleitamento_annual", "energy_fuel_cost_annual", "total_depreciation_annual",
    # --- Resultado econômico --- #
    "gross_margin_annual", "net_margin_annual", "profit_annual", "rcma_annual", "rcma_lactating_cow_day_annual",
    "turnover_rate_annual", "profitability_annual",
    "capital_return_rate_no_land_annual", "capital_return_rate_annual", "capital_return_rate_annual_unfiltered",
    "operating_coverage_point_annual", "total_coverage_point_annual",
    # --- Estoque de capital --- #
    "total_capital_stock_milk_daily_annual", "total_capital_stock_lactating_cow_annual",
    "capital_stock_benfeitorias_share_annual", "capital_stock_maquinas_e_equipamentos_share_annual",
    "capital_stock_animais_share_annual", "capital_stock_terra_share_annual",
    "capital_stock_forrageira_annual_average", "capital_stock_forrageira_share_annual", "forrageira_depreciation_annual",
    # --- Investimento (sem fonte de dados ainda) --- #
    "investment_animals_annual", "investment_machinery_annual", "investment_improvements_annual",
    "investment_land_annual", "investment_total_annual",
    "investment_animals_capital_stock_annual", "investment_machinery_capital_stock_annual",
    "investment_improvements_capital_stock_annual", "investment_total_capital_stock_annual",
    "investment_revenue_annual", "investment_gross_margin_annual",
    # --- Auxiliares sem dado disponível --- #
    "consultant_name", "agroindustry_code",
    "filter_1", "filter_2", "filter_3", "filter_4", "monthly_status_annual",
    "lab_tests", "milk_quality_notes", "outsourcing", "transport", "vaccines",
]
# Mantém apenas as colunas que de fato existem em df_anuais (evita KeyError se alguma dependência faltar).
COLUNAS_RESULTADO_ANUAL = [c for c in COLUNAS_RESULTADO_ANUAL if c in df_anuais.columns]

# Adiciona a grade sistemática de RB/COE/COT/CT/Margens/Lucro por litro, %, VL, área etc.
for prefixo in MEDIDAS_ECONOMICAS_ANUAIS.values():
    for sufixo in [
        "liter_annual", "milk_revenue_percentage_annual", "milk_equivalent_liters_annual",
        "lactating_cow_annual", "total_cows_annual", "hectare_activity_annual", "hectare_activity_com_reserva_annual",
    ]:
        nome_coluna = f"{prefixo}_{sufixo}"
        if nome_coluna in df_anuais.columns and nome_coluna not in COLUNAS_RESULTADO_ANUAL:
            COLUNAS_RESULTADO_ANUAL.append(nome_coluna)

COLUNAS_RESULTADO_ANUAL += [f"{c}_annual_weighted_average" for c in COLUNAS_QUALIDADE if f"{c}_annual_weighted_average" in df_anuais.columns]
COLUNAS_DIMENSAO_ANUAL = [c for c in ["property_name", "labor_rural_code", "entrepreneur_name", "agroindustry_name", "dairy_region", "property_status"] if c in df_anuais.columns]
df_indicadores_anuais = df_anuais[COLUNAS_DIMENSAO_ANUAL + COLUNAS_RESULTADO_ANUAL].copy()

# Rótulo combinado fazenda - produtor, equivalente ao antigo "fazenda-produtor".
if {"property_name", "entrepreneur_name"}.issubset(df_indicadores_anuais.columns):
    df_indicadores_anuais.insert(
        0,
        "property_entrepreneur_label",
        df_indicadores_anuais["property_name"].fillna("") + " - " + df_indicadores_anuais["entrepreneur_name"].fillna(""),
    )

print(f"Colunas no indicador anual: {df_indicadores_anuais.shape[1]:,}")
display(df_indicadores_anuais.head())

#### Renomear colunas

In [ ]:
# ============================================================
# COMPATIBILIDADE COM O "CALCULO_MEDIAS.XLSX" DO NOTEBOOK ANTIGO
# ============================================================
# Rótulo combinado fazenda - produtor diretamente em df_anuais (mesma lógica
# já usada para df_indicadores_anuais), para poder ser puxado pelo mapa abaixo.
if {"property_name", "entrepreneur_name"}.issubset(df_anuais.columns) and "property_entrepreneur_label" not in df_anuais.columns:
    df_anuais["property_entrepreneur_label"] = (
        df_anuais["property_name"].fillna("") + " - " + df_anuais["entrepreneur_name"].fillna("")
    )

# Mesma ordenação usada na exportação técnica, para que a planilha com nomes
# antigos saia com propriedade/período em ordem crescente.
df_anuais = df_anuais.sort_values(["id_property", "annual_period_end"]).reset_index(drop=True)

# Mapa (nome técnico antigo -> nome técnico atual em df_anuais).
# None indica que o indicador ainda não tem fonte de dados no modelo novo
# (Investimento, Estoque de capital em forrageiras, Consultor etc.) — a
# coluna final é criada mesmo assim, vazia, para manter a mesma estrutura
# do arquivo antigo até que essas fontes existam.
MAPA_NOME_ANTIGO_PARA_ATUAL = {
    "idFazenda": "id_property",
    "fazenda-produtor": "property_entrepreneur_label",
    "codAgroindustria": "labor_rural_code",
    "regiaoLeiteira": "dairy_region",
    "nomeagroindustria": "agroindustry_name",
    "nomeconsultor": "consultant_name",
    "intervalo_movel": "annual_period",
    "consistenciaAnual": "annual_consistency_status",
    "Status_Mensais": "monthly_status_annual",
    "sistema": "production_system_annual",
    "Filtro1": "filter_1",
    "Filtro2": "filter_2",
    "Filtro3": "filter_3",
    "Filtro4": "filter_4",
    "taxaRetornoCapitalComTerra_Geral": "capital_return_rate_annual_unfiltered",
    "areaAtividadeSemReserva_mediaMovel": "hectares_atividade_annual_average",
    "areaAtividadeReserva_mediaMovel": "hectares_atividade_com_reserva_annual_average",
    "areaPropria_mediaMovel": "hectares_propria_annual_average",
    "areaArrendada_mediaMovel": "hectares_arrendada_annual_average",
    "percentualAreaArrendada": "rented_area_percentage_annual",
    "precoTerraNua_somaMovel": "land_value_per_hectare_annual",
    "qtdeVacasEmLactacao_mediaMovel": "lactating_cows_annual_average",
    "totalVacas_mediaMovel": "total_cows_annual_average",
    "totalAnimais_mediaMovel": "total_cattle_annual_average",
    "quantidade_Concentrado_Minerais_Anual": "concentrate_mineral_consumed_quantity_annual",
    "MDOTotalDiaria_mediaMovel": "total_labor_quantity_annual_average",
    "MDOFamiliarDiaria_mediaMovel": "family_labor_quantity_annual_average",
    "MDOContratadaDiaria_mediaMovel": "hired_labor_quantity_annual_average",
    "CCS_mediaMovel": "ccs_annual_weighted_average",
    "CPP_mediaMovel": "cpp_annual_weighted_average",
    "Gordura_mediaMovel": "fat_annual_weighted_average",
    "Proteina_mediaMovel": "protein_annual_weighted_average",
    "absCCS_somaMovel": "abs_ccs_annual",
    "absCPP_somaMovel": "abs_cpp_annual",
    "absGordura_somaMovel": "abs_fat_annual",
    "absProteina_somaMovel": "abs_protein_annual",
    "Gordura_VL": "fat_lactating_cow_day_annual",
    "Proteina_VL": "protein_lactating_cow_day_annual",
    "VL_TV_100": "lactating_cows_total_cows_annual",
    "VL_TA_100": "lactating_cows_total_cattle_annual",
    "VL_areaSemReserva": "lactating_cows_hectare_activity_annual",
    "VL_areaComReserva": "lactating_cows_hectare_activity_com_reserva_annual",
    "VL_MDO": "lactating_cows_total_labor_annual",
    "leiteProduzido_somaMovel": "milk_produced_annual",
    "consumoLeiteDescartado_somaMovel": "discarded_quantity_annual",
    "producaoDiaria": "milk_daily_annual",
    "producaoDiaria_VL": "milk_lactating_cow_day_annual",
    "producaoDiaria_TV": "milk_daily_total_cows_annual",
    "producaoDiaria_MDO": "milk_total_labor_day_annual",
    "producaoAnual_AreaSemReserva": "milk_hectare_activity_annual",
    "producaoAnual_AreaComReserva": "milk_hectare_activity_com_reserva_annual",
    "producaoAnual_AreaForrageira": "milk_hectare_forrageira_annual",
    "areaForrageira_mediaMovel": "hectares_forrageira_annual_average",
    "def_rendaAtividade_somaMovel": "total_activity_revenue_annual",
    "def_rendaLeite_somaMovel": "total_milk_revenue_annual",
    "precoLeiteAnual": "milk_revenue_liter_annual",
    "precoConcentradoAnual": "concentrate_mineral_price_annual",
    "relacaoTroca": "milk_concentrate_exchange_ratio_annual",
    "estoqueCapital_semTerra_somaMovel": "capital_stock_no_land_annual_average",
    "estoqueCapital_comTerra_somaMovel": "total_capital_stock_annual_average",
    "def_estoqueCapitalBenfeitorias_somaMovel": "monthly_average_capital_stock_benfeitorias_annual",
    "def_estoqueCapitalMaquinas_somaMovel": "monthly_average_capital_stock_maquinas_e_equipamentos_annual",
    "estoqueCapitalAnimais_Mensal_somaMovel": "animal_capital_stock_annual_average",
    "def_estoqueTerra_Mensal_somaMovel": "land_capital_stock_annual_average",
    "estoqueCapitalPlantio_acumulado_somaMovel": "capital_stock_forrageira_annual_average",
    "def_estoqueCapitalBenfeitorias_somaMovel_ECTotalcomTerra": "capital_stock_benfeitorias_share_annual",
    "def_estoqueCapitalMaquinas_somaMovel_ECTotalcomTerra": "capital_stock_maquinas_e_equipamentos_share_annual",
    "estoqueCapitalAnimais_Mensal_somaMovel_ECTotalcomTerra": "capital_stock_animais_share_annual",
    "def_estoqueTerra_Mensal_somaMovel_ECTotalcomTerra": "capital_stock_terra_share_annual",
    "estoqueCapitalPlantio_acumulado_somaMovel_ECTotalcomTerra": "capital_stock_forrageira_share_annual",
    "coe_somaMovel_litrosEquivalente": "coe_milk_equivalent_liters_annual",
    "cotAnual_litrosEquivalente": "cot_milk_equivalent_liters_annual",
    "ctAnual_litrosEquivalente": "ct_milk_equivalent_liters_annual",
    "coe_somaMovel_precoLeite": "coe_milk_revenue_percentage_annual",
    "cotAnual_precoLeite": "cot_milk_revenue_percentage_annual",
    "ctAnual_precoLeite": "ct_milk_revenue_percentage_annual",
    "margemBrutaAnual": "gross_margin_annual",
    "margemBrutaAnual_litro": "gross_margin_liter_annual",
    "margemBrutaAnual_AreaSemReserva": "gross_margin_hectare_activity_annual",
    "margemBrutaAnual_AreaComReserva": "gross_margin_hectare_activity_com_reserva_annual",
    "margemBrutaAnual_VL": "gross_margin_lactating_cow_annual",
    "margemBrutaAnual_TotalVacas": "gross_margin_total_cows_annual",
    "margemLiquidaAnual": "net_margin_annual",
    "margemLiquidaAnual_litro": "net_margin_liter_annual",
    "margemLiquidaAnual_AreaSemReserva": "net_margin_hectare_activity_annual",
    "margemLiquidaAnual_AreaComReserva": "net_margin_hectare_activity_com_reserva_annual",
    "margemLiquidaAnual_VL": "net_margin_lactating_cow_annual",
    "margemLiquidaAnual_TotalVacas": "net_margin_total_cows_annual",
    "lucroAnual": "profit_annual",
    "lucroAnual_litro": "profit_liter_annual",
    "RCMA": "rcma_annual",
    "RCMA_VL": "rcma_lactating_cow_day_annual",
    "rbl_rba": "rbl_rba_percentage_annual",
    "estoqueCapital_comTerra_somaMovel_VL": "total_capital_stock_lactating_cow_annual",
    "estoqueCapital_comTerra_somaMovel_litro": "total_capital_stock_milk_daily_annual",
    "investimentoAnimais_ECAnimais": "investment_animals_capital_stock_annual",
    "investimentoMaquinas_ECMaquinas": "investment_machinery_capital_stock_annual",
    "investimentoBenfeitorias_ECBenfeitorias": "investment_improvements_capital_stock_annual",
    "investimento_ECcomTerra": "investment_total_capital_stock_annual",
    "investimento_RBA": "investment_revenue_annual",
    "investimento_MB": "investment_gross_margin_annual",
    "investimentoAnimais_somaMovel": "investment_animals_annual",
    "investimentoMaquinas_somaMovel": "investment_machinery_annual",
    "investimentoBenfeitorias_somaMovel": "investment_improvements_annual",
    "investimento_somaMovel": "investment_total_annual",
    "taxadegiro": "turnover_rate_annual",
    "lucratividade": "profitability_annual",
    "taxaRetornoCapitalSemTerra": "capital_return_rate_no_land_annual",
    "taxaRetornoCapitalComTerra": "capital_return_rate_annual",
    "pcot": "operating_coverage_point_annual",
    "pct": "total_coverage_point_annual",
    "coe_somaMovel": "coe_activity_annual",
    "def_gastoAcessoriosDespesasGerais_somaMovel": "general_expenses_annual",
    "def_aleitaento_somaMovel": "aleitamento_annual",
    "def_gastoArrendamento_somaMovel": "land_lease_annual",
    "custoAlimentacao_Concentrado_Minerais_somaMovel": "concentrate_mineral_cost_annual",
    "custoReceitasConcentrado_somaMovel": "concentrated_sold_annual",
    "def_gastoAdministrativo_somaMovel": "administration_annual",
    "def_gastoAssistenciaTecnica_somaMovel": "technical_assistance_annual",
    "custoEnergiaCombustivel": "energy_fuel_cost_annual",
    "Exames_Laboratoriais": "lab_tests",
    "def_gastoHormonios_somaMovel": "hormones_annual",
    "def_gastoImpostoTaxas_somaMovel": "taxes_fees_annual",
    "custoMDOContratada_somaMovel": "hired_labor_expenses_annual",
    "def_gastoMaterialOrdenha_somaMovel": "milking_material_annual",
    "def_gastoMedicamentosVacinas_somaMovel": "medicines_vaccines_annual",
    "Qualidade_Leite": "milk_quality_notes",
    "def_gastoReparosConsertos_somaMovel": "repairs_annual",
    "def_gastoReproducao_somaMovel": "reproduction_annual",
    "Terceirizacao": "outsourcing",
    "Transporte": "transport",
    "Vacinas": "vaccines",
    "custoAlimentacao_Volumoso_somaMovel": "voluminous_amount_total_annual",
    "custoReceitasVolumoso_somaMovel": "voluminous_sold_annual",
    "cotAnual": "cot_annual",
    "depreciacaoEstoqueCapital_somaMovel": "total_depreciation_annual",
    "def_familiarValortotal_somaMovel": "family_labor_expenses_annual",
    "ctAnual": "ct_annual",
    "custoOportunidadeCapital": "capital_opportunity_cost_annual",
}

# ============================================================
# ORDEM E NOMES ANTIGOS (idênticos ao colunas_finais/novos_nomes do notebook antigo)
# ============================================================
COLUNAS_ANTIGAS_EM_ORDEM = [
    "idFazenda", "fazenda-produtor", "codAgroindustria", "regiaoLeiteira", "nomeagroindustria",
    "nomeconsultor", "intervalo_movel", "consistenciaAnual", "Status_Mensais", "sistema",
    "Filtro1", "Filtro2", "Filtro3", "Filtro4", "taxaRetornoCapitalComTerra_Geral",
    "areaAtividadeSemReserva_mediaMovel", "areaAtividadeReserva_mediaMovel", "areaPropria_mediaMovel",
    "areaArrendada_mediaMovel", "percentualAreaArrendada", "precoTerraNua_somaMovel",
    "qtdeVacasEmLactacao_mediaMovel", "totalVacas_mediaMovel", "totalAnimais_mediaMovel",
    "quantidade_Concentrado_Minerais_Anual",
    "MDOTotalDiaria_mediaMovel", "MDOFamiliarDiaria_mediaMovel", "MDOContratadaDiaria_mediaMovel",
    "CCS_mediaMovel", "CPP_mediaMovel", "Gordura_mediaMovel", "Proteina_mediaMovel", "absCCS_somaMovel",
    "absCPP_somaMovel", "absGordura_somaMovel", "absProteina_somaMovel", "Gordura_VL", "Proteina_VL",
    "VL_TV_100", "VL_TA_100", "VL_areaSemReserva", "VL_areaComReserva", "VL_MDO",
    "leiteProduzido_somaMovel", "consumoLeiteDescartado_somaMovel", "producaoDiaria", "producaoDiaria_VL", "producaoDiaria_TV",
    "producaoDiaria_MDO", "producaoAnual_AreaSemReserva", "producaoAnual_AreaComReserva", "producaoAnual_AreaForrageira", "areaForrageira_mediaMovel",
    "def_rendaAtividade_somaMovel", "def_rendaLeite_somaMovel", "precoLeiteAnual", "def_rendaLeite_somaMovel",
    "precoConcentradoAnual", "relacaoTroca",
    "estoqueCapital_semTerra_somaMovel", "estoqueCapital_comTerra_somaMovel", "def_estoqueCapitalBenfeitorias_somaMovel",
    "def_estoqueCapitalMaquinas_somaMovel", "estoqueCapitalAnimais_Mensal_somaMovel", "def_estoqueTerra_Mensal_somaMovel", "estoqueCapitalPlantio_acumulado_somaMovel",
    "def_estoqueCapitalBenfeitorias_somaMovel_ECTotalcomTerra", "def_estoqueCapitalMaquinas_somaMovel_ECTotalcomTerra", "estoqueCapitalAnimais_Mensal_somaMovel_ECTotalcomTerra",
    "def_estoqueTerra_Mensal_somaMovel_ECTotalcomTerra", "estoqueCapitalPlantio_acumulado_somaMovel_ECTotalcomTerra",
    "coe_somaMovel_litrosEquivalente", "cotAnual_litrosEquivalente", "ctAnual_litrosEquivalente",
    "coe_somaMovel_precoLeite", "cotAnual_precoLeite", "ctAnual_precoLeite",
    "margemBrutaAnual", "margemBrutaAnual_litro", "margemBrutaAnual_AreaSemReserva", "margemBrutaAnual_AreaComReserva", "margemBrutaAnual_VL", "margemBrutaAnual_TotalVacas",
    "margemLiquidaAnual", "margemLiquidaAnual_litro", "margemLiquidaAnual_AreaSemReserva", "margemLiquidaAnual_AreaComReserva", "margemLiquidaAnual_VL", "margemLiquidaAnual_TotalVacas",
    "lucroAnual", "lucroAnual_litro",
    "RCMA", "RCMA_VL",
    "rbl_rba",
    "estoqueCapital_comTerra_somaMovel_VL", "estoqueCapital_comTerra_somaMovel_litro",
    "investimentoAnimais_ECAnimais", "investimentoMaquinas_ECMaquinas", "investimentoBenfeitorias_ECBenfeitorias", "investimento_ECcomTerra",
    "investimento_RBA", "investimento_MB", "investimentoAnimais_somaMovel", "investimentoMaquinas_somaMovel", "investimentoBenfeitorias_somaMovel", "investimento_somaMovel",
    "taxadegiro", "lucratividade", "taxaRetornoCapitalSemTerra", "taxaRetornoCapitalComTerra", "pcot", "pct",
    "coe_somaMovel", "def_gastoAcessoriosDespesasGerais_somaMovel", "def_aleitaento_somaMovel", "def_gastoArrendamento_somaMovel",
    "custoAlimentacao_Concentrado_Minerais_somaMovel", "custoReceitasConcentrado_somaMovel", "def_gastoAdministrativo_somaMovel",
    "def_gastoAssistenciaTecnica_somaMovel", "custoEnergiaCombustivel", "Exames_Laboratoriais", "def_gastoHormonios_somaMovel", "def_gastoImpostoTaxas_somaMovel",
    "custoMDOContratada_somaMovel", "def_gastoMaterialOrdenha_somaMovel", "def_gastoMedicamentosVacinas_somaMovel", "Qualidade_Leite", "def_gastoReparosConsertos_somaMovel",
    "def_gastoReproducao_somaMovel", "Terceirizacao", "Transporte", "Vacinas", "custoAlimentacao_Volumoso_somaMovel", "custoReceitasVolumoso_somaMovel",
    "cotAnual", "depreciacaoEstoqueCapital_somaMovel", "def_familiarValortotal_somaMovel", "ctAnual", "custoOportunidadeCapital",
]

NOMES_PORTUGUES_EM_ORDEM = [
    "IDFazenda", "Fazenda - Produtor", "Código LR", "Região", "Agroindústria",
    "Consultor", "Período", "Status - Ind. Anuais", "Status - Ind. Mensais", "Sistema de produção atual",
    "Filtro 1", "Filtro 2", "Filtro 3", "Filtro 4", "AUXILIAR PARA ESTRATIFICAÇÃO - TRCCT CALCULADA",
    "Área destinada à atividade (hectare)", "Área destinada à atividade considerando reserva (hectare)", "Área própria considerando reserva (hectare)",
    "Área arrendada considerando reserva (hectare)", "Percentual de área arrendada (%)", "Preço médio da terra própria (R$/hectare)",
    "Vacas em lactação (animais/mês)", "Total de vacas (animais/mês)", "Total de animais (animais/mês)",
    "Consumo de concentrado anual (Kg/Ano)",
    "Mão de obra total (trabalhador)", "Mão de obra familiar (trabalhador)", "Mão de obra contratada (trabalhador)",
    "CCS (Contagem de células somáticas) (x1000 células/ml)", "CPP (Contagem padrão em placas) (x1000 UFC/ml)", "Gordura (%)", "Proteína (%)",
    "AUXILIAR PARA MÉDIAS - CCS (Contagem de células somáticas) (x1000 células/ml)", "AUXILIAR PARA MÉDIAS - CPP (Contagem padrão em placas) (x1000 UFC/ml)",
    "AUXILIAR PARA MÉDIAS - Gordura (%)", "AUXILIAR PARA MÉDIAS - Proteína (%)", "Gordura (kg/vaca em lactação/dia)", "Proteína (kg/vaca em lactação/dia)",
    "Vacas em lactação/total de vacas (%)", "Vacas em lactação/total de animais (%)",
    "Vacas em lactação/área destinada à atividade (animais/hectare)", "Vacas em lactação/área destinada à atividade considerando reserva (animais/hectare)",
    "Vacas em lactação/mão de obra total (animais/trabalhador/dia)",
    "Produção anual de leite (litros/ano)", "Leite descartado (litros/ano)", "Produção diária de leite (litros/dia)",
    "Produção/vacas em lactação (litros/animal/dia)", "Produção/total de vacas (litros/animal/dia)",
    "Produção/mão de obra total (litros/trabalhador/dia)", "Produção/área destinada à atividade (litros/hectare/ano)",
    "Produção/área destinada à atividade considerando reserva (litros/hectare/ano)", "Produção / área de produção de forrageira (litros/hectare/ano)",
    "AUXILIAR PARA MÉDIAS - Produção / área de produção de forrageira (litros/hectare/ano)",
    "Renda bruta da atividade leiteira (R$/ano)", "Renda bruta do leite (R$/ano)", "Preço médio do leite (R$/litro)",
    "AUXILIAR PARA MÉDIAS - Preço médio do leite (R$/litro)", "Preço médio do concentrado (R$/Kg)", "Relação de troca leite/concentrado (Kg/L)",
    "Estoque de capital total sem terra (R$)", "Estoque de capital total com terra (R$)", "Estoque de capital em benfeitorias (R$)",
    "Estoque de capital em máquinas (R$)", "Estoque de capital em animais (R$)", "Estoque de capital em terra (R$)", "Estoque de capital em forrageiras não-anuais (R$)",
    "Estoque de capital em benfeitorias/estoque de capital total com terra (%)", "Estoque de capital em máquinas/estoque de capital total com terra (%)",
    "Estoque de capital em animais/estoque de capital total com terra (%)", "Estoque de capital em terra/estoque de capital total com terra (%)",
    "Estoque de capital em forrageiras não-anuais/estoque de capital total com terra (%)",
    "COE da atividade leiteira em equivalentes litros de leite (litros/ano)", "COT da atividade leiteira em equivalentes litros de leite (litros/ano)",
    "CT da atividade leiteira em equivalentes litros de leite (litros/ano)",
    "COE do leite/preço do leite (%)", "COT do leite/preço do leite (%)", "CT do leite/preço do leite (%)",
    "Margem bruta da atividade (R$/ano)", "Margem bruta unitária (R$/litro)", "Margem bruta/área destinada à atividade (R$/hectare/ano)",
    "Margem bruta/área destinada à atividade considerando reserva (R$/hectare/ano)", "Margem bruta/vacas em lactação (R$/animal/ano)", "Margem bruta/total de vacas (R$/animal/ano)",
    "Margem líquida da atividade (R$/ano)", "Margem líquida unitária (R$/litro)", "Margem líquida/área destinada à atividade (R$/hectare/ano)",
    "Margem líquida/área destinada à atividade considerando reserva (R$/hectare/ano)", "Margem líquida/vacas em lactação (R$/animal/ano)", "Margem líquida/total de vacas (R$/animal/ano)",
    "Lucro total (R$/ano)", "Lucro unitário (R$/litro)",
    "RMCA (Receita Menos Custo com Alimentação) (R$/ano)", "RMCA (Receita Menos Custo com Alimentação) (R$/vaca em lactação/dia)",
    "Renda do leite/renda atividade (%)",
    "Estoque de capital total com terra/vaca em lactação (R$/animal)", "Estoque de capital total com terra/produção diária de leite (R$/litro/dia)",
    "Investimento em animais/estoque de capital em animais (%)", "Investimento em máquinas e equipamento/estoque de capital em máquinas e equipamentos (%)",
    "Investimento em benfeitorias/estoque de capital em benfeitorias (%)", "Investimento anual/estoque de capital total com terra (%)",
    "Investimento anual/renda bruta da atividade (%)", "Investimento anual/margem bruta (%)",
    "AUXILIAR PARA MÉDIAS - Investimento em animais", "AUXILIAR PARA MÉDIAS - Investimento em máquinas e equipamento",
    "AUXILIAR PARA MÉDIAS - Investimento em benfeitorias", "AUXILIAR PARA MÉDIAS - Investimento anual",
    "Taxa de giro do estoque de capital total (%)", "Lucratividade operacional (%)",
    "Taxa de remuneração do capital sem terra (% ao ano)", "Taxa de remuneração do capital com terra (% ao ano)",
    "Ponto de cobertura operacional total da atividade (litros/dia)", "Ponto de cobertura total da atividade (litros/dia)",
    "Custo operacional efetivo - R$/ano (Atividade)", "Acessórios e despesas em geral - R$/ano (Atividade)", "Aleitamento - R$/ano (Atividade)",
    "Arrendamento/Aluguel - R$/ano (Atividade)", "Concentrados e Minerais de ingestão livre - R$/ano (Atividade)", "Concentrados (Vendido) - R$/ano (Atividade)",
    "Despesas administrativas - R$/ano (Atividade)", "Despesas com assistência técnica - R$/ano (Atividade)", "Energia e combustível - R$/ano (Atividade)",
    "Exames laboratoriais - R$/ano (Atividade)", "Hormônios - R$/ano (Atividade)", "Impostos e taxas - R$/ano (Atividade)",
    "Mão de obra contratada - R$/ano (Atividade)", "Material de ordenha - R$/ano (Atividade)", "Medicamentos - R$/ano (Atividade)",
    "Qualidade do leite - R$/ano (Atividade)", "Reparos e consertos de máquinas e benfeitorias - R$/ano (Atividade)", "Reprodução - R$/ano (Atividade)",
    "Terceirização de recria - R$/ano (Atividade)", "Transporte e descontos no leite - R$/ano (Atividade)", "Vacinas - R$/ano (Atividade)",
    "Volumosos - R$/ano (Atividade)", "Volumosos (Vendido) - R$/ano (Atividade)",
    "Custo operacional total - R$/ano (Atividade)", "Depreciação - R$/ano (Atividade)", "Mão de obra familiar - R$/ano (Atividade)",
    "Custo total - R$/ano (Atividade)", "Remuneração do capital - R$/ano (Atividade)",
]

assert len(COLUNAS_ANTIGAS_EM_ORDEM) == len(NOMES_PORTUGUES_EM_ORDEM) == 140, (
    "As listas de nomes antigos e novos precisam ter 140 posições cada, igual ao notebook original."
)

# ============================================================
# MONTAR A TABELA COM OS NOMES/ORDEM DO CALCULO_MEDIAS.XLSX ANTIGO
# ============================================================
dados_renomeados = {}
colunas_sem_correspondencia = []

for nome_antigo, nome_pt in zip(COLUNAS_ANTIGAS_EM_ORDEM, NOMES_PORTUGUES_EM_ORDEM):

    nome_atual = MAPA_NOME_ANTIGO_PARA_ATUAL.get(nome_antigo)

    if nome_atual is not None and nome_atual in df_anuais.columns:
        valores = df_anuais[nome_atual]
    else:
        valores = pd.Series([None] * len(df_anuais), index=df_anuais.index)
        colunas_sem_correspondencia.append((nome_antigo, nome_pt))

    # "AUXILIAR PARA MÉDIAS - Preço médio do leite" repete a mesma coluna de origem
    # que "Renda bruta do leite (R$/ano)" — isso é intencional (auxiliar de recálculo
    # de médias ponderadas no BI), igual ao notebook antigo.
    if nome_pt in dados_renomeados:
        nome_pt = f"{nome_pt} (2)"

    dados_renomeados[nome_pt] = valores.values

df_calculo_medias = pd.DataFrame(dados_renomeados, index=df_anuais.index)

print(f"Colunas mapeadas com dado real: {140 - len(colunas_sem_correspondencia)} / 140")
print(f"Colunas sem correspondência no modelo novo (ficam vazias): {len(colunas_sem_correspondencia)}")
for nome_antigo, nome_pt in colunas_sem_correspondencia:
    print(f"  - {nome_antigo} -> {nome_pt}")

display(df_calculo_medias.head())

#### Exportar Indicadores Anuais

In [ ]:
DATA_EXPORTACAO = datetime.now().strftime("%Y_%m_%d")
PASTA_SAIDA = Path.cwd().parent / "data" / "outputs" / "annual"
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
CAMINHO_ANUAIS = PASTA_SAIDA / f"{DATA_EXPORTACAO}_indicadores_anuais.xlsx"
df_exportacao_anual = preparar_dataframe_para_excel(df_indicadores_anuais)
df_exportacao_anual = df_exportacao_anual.sort_values(["id_property", "annual_period_end"]).reset_index(drop=True)

# Aba extra com os mesmos nomes/ordem de colunas do antigo calculo_medias.xlsx.
df_exportacao_calculo_medias = preparar_dataframe_para_excel(df_calculo_medias)

# Aba extra com o relatório de cobertura de lançamento de dados (separado dos indicadores).
df_exportacao_cobertura = preparar_dataframe_para_excel(df_cobertura_anual)

exportar_varias_abas_xlsx(
    abas={
        "Indicadores Anuais": df_exportacao_anual,
        "Cálculo de Médias": df_exportacao_calculo_medias,
        "Cobertura de Dados": df_exportacao_cobertura,
    },
    caminho_saida=CAMINHO_ANUAIS,
    fonte="Aptos",
)

print(f"Linhas anuais exportadas: {len(df_exportacao_anual):,}")
print(f"Arquivo: {CAMINHO_ANUAIS}")